# RAG-Adapter research notebook
See `docs/notebooks.md` before running individual experiment sections. Outputs are cleared and local paths use the configuration cell below.


In [ ]:
from __future__ import annotations
import os
from pathlib import Path
RELEASE_ROOT = Path.cwd().resolve()
if RELEASE_ROOT.name == "notebooks":
    RELEASE_ROOT = RELEASE_ROOT.parent
DATA_ROOT = str(Path(os.environ.get("RAG_DATA_ROOT", RELEASE_ROOT / "data_local")).expanduser().resolve())
WORK_ROOT = str(Path(os.environ.get("RAG_WORK_ROOT", RELEASE_ROOT / "runs")).expanduser().resolve())
BGE_MODEL = os.environ.get("RAG_BGE_MODEL", "BAAI/bge-m3")
Path(WORK_ROOT).mkdir(parents=True, exist_ok=True)


# Loading Python libraries

In [ ]:
import os
import umap
import time
import csv
import pysrt
import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import random
import numpy as np
import itertools
import shutil
from openai import OpenAI
import pyarrow.parquet as pq
from collections import defaultdict, Counter
import base64
from PIL import Image
import matplotlib.pyplot as plt
import qdrant_client
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.indices import MultiModalVectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.base.embeddings.base import BaseEmbedding
from llama_index.core.embeddings.utils import resolve_embed_model

from llama_index.finetuning.embeddings.common import (
    EmbeddingQAFinetuneDataset,
)
from llama_index.finetuning.types import BaseEmbeddingFinetuneEngine
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from llama_index.core.schema import ImageNode
from collections import defaultdict
from llama_index.core import (
            Settings,
            StorageContext,
            VectorStoreIndex,
            SimpleDirectoryReader,
            load_index_from_storage,
        )

from typing import Dict, List, Tuple, Any, Iterable, Optional
from llama_index.core.schema import MetadataMode, TextNode
from tqdm import tqdm
import uuid
import json
import clip

from llama_index.core.response.notebook_utils import display_source_node

import torch
from torch import Tensor, nn
from sentence_transformers import util
from sentence_transformers.SentenceTransformer import SentenceTransformer
from llama_index.embeddings.clip import ClipEmbedding

from torch.optim import lr_scheduler    
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup

from llama_index.finetuning import generate_qa_embedding_pairs, SentenceTransformersFinetuneEngine, EmbeddingQAFinetuneDataset

import logging
from llama_index.core.base.embeddings.base import Embedding
from llama_index.core.bridge.pydantic import Field, PrivateAttr
from llama_index.core.constants import DEFAULT_EMBED_BATCH_SIZE
from llama_index.core.embeddings.multi_modal_base import MultiModalEmbedding
from llama_index.core.schema import ImageType
from llama_index.core.schema import Document

from time import strftime
from time import gmtime


# Loading the Video-MME Q&A

In [ ]:
# Load test problem
data = pq.ParquetFile(f"{DATA_ROOT}/dataset/Video-MME/test-00000-of-00001.parquet")
table = data.read()

questions_video_mme = defaultdict()

video_ids = []
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Video-MME/frames"):
    if len(ds) == 0:
        video_id = root.split("/")[-1]
        video_ids.append(video_id)


for i in range(data.num_row_groups):
    row_group = data.read_row_group(i)
    row_group = row_group.to_pandas()
    for idx, row in row_group.iterrows():
        if row.videoID in video_ids:
            # print(row.question_id)
            if row.videoID not in questions_video_mme:
                questions_video_mme[row.videoID] = {}
            if row.question_id not in questions_video_mme[row.videoID]:
                questions_video_mme[row.videoID][row.question_id] = {}
            questions_video_mme[row.videoID][row.question_id]["question"] = row.question
            questions_video_mme[row.videoID][row.question_id]["options"] = row.options
            questions_video_mme[row.videoID][row.question_id]["answer"] = row.answer
            questions_video_mme[row.videoID][row.question_id]["duration"] = row.duration
            questions_video_mme[row.videoID][row.question_id]["domain"] = row.domain


# Load the MLVU Q&A

In [ ]:
tasks = ['1_plotQA', '2_needle', '3_ego', '4_count', '5_order', '6_anomaly_reco', '7_topic_reasoning', '8_sub_scene', '9_summary']
questions_mlvu = defaultdict(dict)
video_ids = defaultdict(list)
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MLVU/frames"):
    if len(ds) == 0:
        task = root.split("/")[-2]
        video_id = root.split("/")[-1]
        video_ids[task].append(video_id)
        
for task in tasks:
    file = os.path.join(f"{DATA_ROOT}/dataset/MLVU", f"MLVU_json_{task}.json")
    with open(file, 'r', encoding='utf-8') as f:
        qs = json.load(f)
    for q in qs:
        video_id = q["video"].split(".")
        video_id = video_id[0]
        if video_id not in video_ids[task]:
            continue
        if video_id not in questions_mlvu:
            questions_mlvu[video_id]["question"] = []
            questions_mlvu[video_id]["answer"] = []
            questions_mlvu[video_id]["candidates"] = []
        if "candidates" in q:
            questions_mlvu[video_id]["candidates"].append(q["candidates"])
        questions_mlvu[video_id]["question"].append(q["question"])
        questions_mlvu[video_id]["answer"].append(q["answer"])
        questions_mlvu[video_id]["duration"] = q["duration"]
        questions_mlvu[video_id]["question_type"] = task
# print(len(questions))


# Load Perception Test Q&A

In [ ]:
with open(f"{DATA_ROOT}/dataset/Perception_Test/mc_question_train.json", "r") as json_file:
    mc_question = json.load(json_file)

questions_perception_test = defaultdict()
alts = ["A", "B", "C"]
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Perception_Test/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]
            if video_id not in questions_perception_test:
                questions_perception_test[video_id] = defaultdict(list)
            for q in mc_question[video_id]["mc_question"]:
                questions_perception_test[video_id]["id"].append(q["id"])
                questions_perception_test[video_id]["question"].append(q["question"])
                index_options = []
                for i, opt in enumerate(q["options"]):
                    index_options.append(f"{alts[i]}. {opt}")
                questions_perception_test[video_id]["options"].append(index_options)
                questions_perception_test[video_id]["answer_id"].append(alts[int(q["answer_id"])])

# print(video_id)
# for i in range(len(questions_perception_test[video_id]["question"])):
#     question = ""
#     question += questions_perception_test[video_id]["question"][i] + "\n"
#     question += '\n'.join(questions_perception_test[video_id]["options"][i]) + "\n"
#     print(question)


# Load Schema Q&A

In [ ]:
with open(f"{DATA_ROOT}/dataset/EgoSchema/questions.json", "r") as q_file:
    mc_questions = json.load(q_file)

with open(f"{DATA_ROOT}/dataset/EgoSchema/subset_answers.json", "r") as a_file:
    mc_answers = json.load(a_file)

questions_egoschema = defaultdict()
alts = ["A", "B", "C", "D", "E"]
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/EgoSchema/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]
            if video_id not in questions_egoschema:
                questions_egoschema[video_id] = defaultdict(list)
            for video in mc_questions:

                if video["q_uid"] == video_id:
                    questions_egoschema[video_id]["question"].append(video["question"])

                    index_options = []
                    index_options.append(f"A. {video['option 0']}")
                    index_options.append(f"B. {video['option 1']}")
                    index_options.append(f"C. {video['option 2']}")
                    index_options.append(f"D. {video['option 3']}")
                    index_options.append(f"E. {video['option 4']}")

                    questions_egoschema[video_id]["options"].append(index_options)
                    questions_egoschema[video_id]["answer_id"].append(alts[int(mc_answers[video_id])])


# Post-process the fine-tuned data set, locate abnormal generated content and regenerate captions

In [ ]:
# Calculate the proportion of each character in captions, and filter documents that contain characters with high repetition rates
def character_proportion(text):
    char_count = Counter(text)
    total_chars = len(text)
    char_proportions = {char: count / total_chars for char, count in char_count.items()}
    return char_proportions

finetune_datasets = ['MSVD-QA','MSRVTT-QA','ActivityNet-QA','TGIF-QA']
for dataset in finetune_datasets:
    for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/{dataset}/captions"):
        if len(ds) == 0:
            if len(fs) != 0:
                for file in fs:
                    file_path = os.path.join(root, file)
                    with open(file_path, 'r', encoding='utf-8') as f:
                        captions = f.read()
                    captions = captions.replace(" ", "").lower()
                    proportions = character_proportion(captions)
                    for char, proportion in proportions.items():
                        if proportion > 0.2:
                            print(f'Exception: {file_path}')
                            break
                        # print(f"'{char}': {proportion:.2%}")


# Import the fine-tuning data set

In [ ]:
train_captions = []
test_captions = []
train_questions_dict = defaultdict(list)
test_questions_dict = defaultdict(list)

# Treat train and val of all data sets as the training data and test as the test data

# MSVD-QA
msvd_id2youtube = defaultdict()
with open(f"{DATA_ROOT}/dataset/MSVD-QA/youtube_mapping.txt", 'r', encoding='utf-8') as mapping:
    text = mapping.read()
text_split = text.split("\n")
for t in text_split:
    youtube_id = t.split(" ")[0]
    vid = t.split(" ")[1]
    vid_num = vid[3:]
    msvd_id2youtube[vid_num] = youtube_id

splits_msvd = ['train', 'val', 'test']
for split in splits_msvd:
    json_path = f"{DATA_ROOT}/dataset/MSVD-QA/MSVD-QA/{split}_qa.json"
    file = open(json_path)
    contents = [json.loads(line) for line in file][0]
    for content in contents:
        vid_num = str(content["video_id"])
        question = content["question"]
        if split == "train" or split == "val":
            train_questions_dict[msvd_id2youtube[vid_num]].append(question)
        else:
            test_questions_dict[msvd_id2youtube[vid_num]].append(question)

for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MSVD-QA/captions"):
    if len(ds) == 0:
        youtube_id = os.path.basename(root)
        frames_num = os.listdir(os.path.join(f"{DATA_ROOT}/dataset/MSVD-QA/frames", youtube_id))
        if len(frames_num) > 300:
            continue
        if len(fs) != 0:
            if youtube_id in train_questions_dict:
                train_captions.extend(SimpleDirectoryReader(root).load_data())
            if youtube_id in test_questions_dict:
                test_captions.extend(SimpleDirectoryReader(root).load_data())

# Total 1910 videos (1970 videos in the original dataset, 60 less videos in the json file)
# print(f"MSVD-QA len(train_captions): {len(train_captions)}")
# print(f"MSVD-QA len(test_captions): {len(test_captions)}")
# print(f"MSVD-QA len(train_questions_dict): {len(train_questions_dict)}")
# print(f"MSVD-QA len(test_questions_dict): {len(test_questions_dict)}")
                    
                
# # MSRVTT-QA
splits_msvd = ['train', 'val', 'test']
for split in splits_msvd:
    json_path = f"{DATA_ROOT}/dataset/MSRVTT-QA/MSRVTT-QA/{split}_qa.json"
    file = open(json_path)
    contents = [json.loads(line) for line in file][0]
    for content in contents:
        vid_num = "video" + str(content["video_id"])
        question = content["question"]
        if split == "train" or split == "val":
            train_questions_dict[vid_num].append(question)
        else:
            test_questions_dict[vid_num].append(question)


for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MSRVTT-QA/captions"):
    if len(ds) == 0:
        video_id = os.path.basename(root)
        frames_num = os.listdir(os.path.join(f"{DATA_ROOT}/dataset/MSRVTT-QA/frames", video_id))
        if len(frames_num) > 300:
            continue
        if len(fs) != 0:
            if video_id in train_questions_dict:
                train_captions.extend(SimpleDirectoryReader(root).load_data())
            if video_id in test_questions_dict:
                test_captions.extend(SimpleDirectoryReader(root).load_data())

# # print(f"MSRVTT-QA len(train_captions): {len(train_captions)}")
# # print(f"MSRVTT-QA len(test_captions): {len(test_captions)}")
# # print(f"MSRVTT-QA len(train_questions_dict): {len(train_questions_dict)}")
# # print(f"MSRVTT-QA len(test_questions_dict): {len(test_questions_dict)}")
                
# # ActivityNet-QA
splits_activitynet = ['train', 'val', 'test']
for split in splits_activitynet:
    json_path = f"{DATA_ROOT}/dataset/ActivityNet-QA/{split}_q.json"
    file = open(json_path)
    contents = [json.loads(line) for line in file][0]
    for content in contents:
        video_id = "v_" + content["video_name"]
        question = content["question"]
        if split == "train" or split == "val":
            train_questions_dict[video_id].append(question)
        else:
            test_questions_dict[video_id].append(question)

video_chatgpt_instruction_path = f"{DATA_ROOT}/dataset/ActivityNet-QA/video_chatgpt_instruct.json"
instruction_json_file = open(video_chatgpt_instruction_path)
instruction_contents = [json.loads(line) for line in instruction_json_file][0]
for instruction in instruction_contents:
    video_id = instruction["video_id"]
    question = instruction["q"]
    train_questions_dict[video_id].append(question)

for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/ActivityNet-QA/captions"):
    if len(ds) == 0:
        video_id = os.path.basename(root)
        frames_num = os.listdir(os.path.join(f"{DATA_ROOT}/dataset/ActivityNet-QA/frames", video_id))
        if len(frames_num) > 300:
            continue
        if len(fs) != 0:
            if video_id in train_questions_dict:
                train_captions.extend(SimpleDirectoryReader(root).load_data())
            if video_id in test_questions_dict:
                test_captions.extend(SimpleDirectoryReader(root).load_data())

# print(f"ActivityNet-QA len(train_captions): {len(train_captions)}")
# print(f"ActivityNet-QA len(test_captions): {len(test_captions)}")
# print(f"ActivityNet-QA len(train_questions_dict): {len(train_questions_dict)}")
# print(f"ActivityNet-QA len(test_questions_dict): {len(test_questions_dict)}")

# TGIF-QA
splits_tgif = ['Train', 'Test']
type_tgif = ['action', 'count', 'frameqa', 'transition']
def read_csv_with_csv_module(file_path, dicts):
    with open(file_path, mode='r', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile, delimiter='+') 
        for row in reader:
            for k, y in row.items():
                y_content = y.split('\t')
                dicts[y_content[0]].append(y_content[1])

for split in splits_tgif:
    for type in type_tgif:
        csv_path = f"{DATA_ROOT}/dataset/TGIF-QA/{split}_{type}_question.csv"
        if split == "Train":
            read_csv_with_csv_module(csv_path, train_questions_dict)
        else:
            read_csv_with_csv_module(csv_path, test_questions_dict)

for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/TGIF-QA/captions"):
    if len(ds) == 0:
        video_id = os.path.basename(root)
        frames_num = os.listdir(os.path.join(f"{DATA_ROOT}/dataset/TGIF-QA/frames", video_id))
        if len(frames_num) > 300:
            continue
        if len(fs) != 0:
            if video_id in train_questions_dict:
                train_captions.extend(SimpleDirectoryReader(root).load_data())
            if video_id in test_questions_dict:
                test_captions.extend(SimpleDirectoryReader(root).load_data())
         
# print(f"TGIF-QA len(train_captions): {len(train_captions)}")
# print(f"TGIF-QA len(test_captions): {len(test_captions)}")
# print(f"TGIF-QA len(train_questions_dict): {len(train_questions_dict)}")
# print(f"TGIF-QA len(test_questions_dict): {len(test_questions_dict)}")

print(f"len(train_captions): {len(train_captions)}")
print(f"len(test_captions): {len(test_captions)}")
print(f"len(train_questions_dict): {len(train_questions_dict)}")
print(f"len(test_questions_dict): {len(test_questions_dict)}")


In [ ]:
def load_existing_data(
    path: str,
) -> Tuple[Dict[str, str], Dict[str, str], Dict[str, List[str]]]:
    """Load existing data from a JSON file if it exists.

    Args:
        path (str): The file path to load the JSON from.

    Returns:
        Tuple[Dict[str, str], Dict[str, str], Dict[str, List[str]]]: The loaded queries, corpus, and relevant_docs.
    """
    try:
        with open(path) as f:
            data = json.load(f)
        return data["queries"], data["corpus"], data["relevant_docs"]
    except FileNotFoundError:
        return {}, {}, {}

def generate_qa_embedding_pairs(
    nodes: List[TextNode],
    questions_dict: Dict[str, list],
    num_questions_per_chunk: int = 2,
    retry_limit: int = 3,
    on_failure: str = "continue",  # options are "fail" or "continue"
    save_every: int = 2000,
    output_path: str = "qa_finetune_dataset.json",
    verbose: bool = True,
) -> EmbeddingQAFinetuneDataset:
    """Generate QA pairs from a set of nodes and save periodically.

    Args:
        nodes (List[TextNode]): List of TextNode objects to process.
        num_questions_per_chunk (int): Number of questions to generate per chunk of text.
        retry_limit (int): Number of times to retry on failure.
        on_failure (str): Action to take on repeated failures ('fail' or 'continue').
        save_every (int): Number of nodes to process before saving the dataset.
        output_path (str): The file path to save the JSON output.
        verbose (bool): If True, print debugging messages.

    Returns:
        EmbeddingQAFinetuneDataset: The generated dataset.
    """
    queries, corpus, relevant_docs = load_existing_data(output_path) # {} {} {}

    node_dict = {
        node.metadata['file_path'].split('/')[-2]: node.get_content(metadata_mode=MetadataMode.NONE)
        for node in nodes
    }

    start_index = len(corpus)

    save_counter = start_index

    for video_id, text in tqdm(
        list(node_dict.items())[start_index:], initial=start_index
    ):
        
        questions = questions_dict[video_id]

        num_questions_generated = len(questions)
        for question in questions:
            question_id = str(uuid.uuid4())
            queries[question_id] = question
            relevant_docs[question_id] = [video_id]

        corpus[video_id] = text

        save_counter += 1
        if save_counter % save_every == 0:
            dataset = EmbeddingQAFinetuneDataset(
                queries=queries, corpus=corpus, relevant_docs=relevant_docs
            )
            dataset.save_json(output_path)
            if verbose:
                print(f"Saved progress at {save_counter} entries.")

    # Save final dataset
    dataset = EmbeddingQAFinetuneDataset(
        queries=queries, corpus=corpus, relevant_docs=relevant_docs
    )
    dataset.save_json(output_path)
    if verbose:
        print("Final dataset saved.")

    return dataset

train_dataset = generate_qa_embedding_pairs(
        nodes=train_captions, 
        questions_dict=train_questions_dict, 
        output_path=f"{WORK_ROOT}/finetune/video_mme/train_dataset.json"
    )

val_dataset = generate_qa_embedding_pairs(
        nodes=test_captions,
        questions_dict=test_questions_dict,
        output_path=f"{WORK_ROOT}/finetune/video_mme/test_dataset.json"
    )


# Load the GPT4o client

In [ ]:
from openai import OpenAI
client = OpenAI()  # Reads OPENAI_API_KEY and optional OPENAI_BASE_URL.

def call_gpt_with_messages(messages, model_name):
    if model_name == "gpt_4o":
        model = "gpt-4o-2024-08-06"
    if model_name == "gpt_4_turbo":
        model = os.environ.get("RAG_GPT4_TURBO_MODEL", "gpt-4-turbo-2024-04-09")
    response = client.chat.completions.create(
        model=model, 
        messages=messages,
        max_tokens=300,
        temperature=0.7,
        )
    return response


# Custom CLIP loading class

In [ ]:
logger = logging.getLogger(__name__)

AVAILABLE_CLIP_MODELS = (
    "RN50",
    "RN101",
    "RN50x4",
    "RN50x16",
    "RN50x64",
    "ViT-B/32",
    "ViT-B/16",
    "ViT-L/14",
    "ViT-L/14@336px",
)
DEFAULT_CLIP_MODEL = "ViT-B/32"

class ClipEmbedding(MultiModalEmbedding):
    """CLIP embedding models for encoding text and image for Multi-Modal purpose.

    This class provides an interface to generate embeddings using a model
    deployed in OpenAI CLIP. At the initialization it requires a model name
    of CLIP.

    Note:
        Requires `clip` package to be available in the PYTHONPATH. It can be installed with
        `pip install git+https://github.com/openai/CLIP.git`.
    """

    embed_batch_size: int = Field(default=DEFAULT_EMBED_BATCH_SIZE, gt=0)

    _clip: Any = PrivateAttr()
    _model: Any = PrivateAttr()
    _preprocess: Any = PrivateAttr()
    _device: Any = PrivateAttr()

    @classmethod
    def class_name(cls) -> str:
        return "ClipEmbedding"

    def __init__(
        self,
        *,
        embed_batch_size: int = DEFAULT_EMBED_BATCH_SIZE,
        model_name: str = DEFAULT_CLIP_MODEL,
        ck_path: str = None,
        **kwargs: Any,
    ):
        """Initializes the ClipEmbedding class.

        During the initialization the `clip` package is imported.

        Args:
            embed_batch_size (int, optional): The batch size for embedding generation. Defaults to 10,
                must be > 0 and <= 100.
            model_name (str): The model name of Clip model.

        Raises:
            ImportError: If the `clip` package is not available in the PYTHONPATH.
            ValueError: If the model cannot be fetched from Open AI. or if the embed_batch_size
                is not in the range (0, 100].
        """
        if embed_batch_size <= 0:
            raise ValueError(f"Embed batch size {embed_batch_size}  must be > 0.")

        try:
            import clip
            import torch
        except ImportError:
            raise ImportError(
                "ClipEmbedding requires `pip install git+https://github.com/openai/CLIP.git` and torch."
            )

        super().__init__(
            embed_batch_size=embed_batch_size, model_name=model_name, **kwargs
        )

        try:
            self._device = "cuda" if torch.cuda.is_available() else "cpu"
            is_local_path = os.path.exists(self.model_name)
            if not is_local_path and self.model_name not in AVAILABLE_CLIP_MODELS:
                raise ValueError(
                    f"Model name {self.model_name} is not available in CLIP."
                )
            self._model, self._preprocess = clip.load(
                self.model_name, device=self._device
            )
            if ck_path is not None:
                self._model.load_state_dict(torch.load(ck_path, map_location=self._device))

        except Exception as e:
            logger.error("Error while loading clip model.")
            raise ValueError("Unable to fetch the requested embeddings model") from e

    # TEXT EMBEDDINGS

    async def _aget_query_embedding(self, query: str) -> Embedding:
        return self._get_query_embedding(query)

    def _get_text_embedding(self, text: str) -> Embedding:
        return self._get_text_embeddings([text])[0]

    def _get_text_embeddings(self, texts: List[str]) -> List[Embedding]:
        results = []
        for text in texts:
            try:
                import clip
            except ImportError:
                raise ImportError(
                    "ClipEmbedding requires `pip install git+https://github.com/openai/CLIP.git` and torch."
                )
            text_embedding = self._model.encode_text(
                clip.tokenize(text).to(self._device)
            )
            results.append(text_embedding.tolist()[0])

        return results

    def _get_query_embedding(self, query: str) -> Embedding:
        return self._get_text_embedding(query)

    # IMAGE EMBEDDINGS

    async def _aget_image_embedding(self, img_file_path: ImageType) -> Embedding:
        return self._get_image_embedding(img_file_path)

    def _get_image_embedding(self, img_file_path: ImageType) -> Embedding:
        import torch

        with torch.no_grad():
            image = (
                self._preprocess(Image.open(img_file_path))
                .unsqueeze(0)
                .to(self._device)
            )
            return self._model.encode_image(image).tolist()[0]


# MMR implementation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def mmr(query_image_scores, selected_items, candidate_items, lambda_param, frame_embeddings, caption_embeddings=None):
    max_score = -float('inf')
    for i in candidate_items:
        for j in selected_items:
            sim = torch.cosine_similarity(frame_embeddings[j], frame_embeddings[i], dim=0)
            if caption_embeddings is not None:
                sim += torch.cosine_similarity(caption_embeddings[j], caption_embeddings[i], dim=0)
            score = lambda_param * query_image_scores[i] - (1 - lambda_param) * sim

            if score > max_score:
                max_score = score
                selected_item = i

    return selected_item

def mmr_selection(query_image_scores, bge_m3, clip, lambda_param=0.7, top_k=10):
    selected_items = []
    candidate_items = list(query_image_scores.keys())

    # store the caption and image embeddings for query_image_scores.items()
    caption_embeddings = {}
    frame_embeddings = {}
    for i in candidate_items:
        caption_path = i.replace("jpg", "txt").replace("frames", "captions")
        with open(caption_path, 'r', encoding='utf-8') as f:
            caption = f.read()
        caption_embeddings[i] = torch.tensor(bge_m3.get_text_embedding(caption), device=device) # list (1024)
        frame_embeddings[i] = torch.tensor(clip._get_image_embedding(i), device=device) # list (768)
    
    for i in range(top_k):
        if i == 0:
            # Select the item with the highest score as the first item
            item = max(candidate_items, key=lambda x: query_image_scores[x])
            selected_items.append(item)
            candidate_items.remove(item)
            continue
        
        # Calculate the margin relevance for each candidate item
        selected_item = mmr(query_image_scores, selected_items, candidate_items, lambda_param, frame_embeddings, caption_embeddings)
        # selected_item = mmr(query_image_scores, selected_items, candidate_items, lambda_param, frame_embeddings, None)
        selected_items.append(selected_item)
        candidate_items.remove(selected_item)
    return selected_items


# Test 1: Uniform Sampling Test (GPT4o)

## Video-MME

In [ ]:
for video_id in picked:
    save_path = f"{DATA_ROOT}/results/Video-MME/gpt_4_turbo_10/no_rag/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    else:
        files = os.listdir(save_path)
        if len(files) == 3:
            continue

    frames = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}")
    frames.sort()
    n_frms = 10

    indices = np.linspace(0, len(frames), num=n_frms, endpoint=False, dtype=int).tolist()
    image_path_list = []
    for i in indices:
        image_path_list.append(f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{frames[i]}")

    # generate questions
    for question_id in questions[video_id]:
        print(f"processing: {video_id} {question_id}")

        question = ""
        question += questions[video_id][question_id]["question"] + "\n"
        question += '\n'.join(questions[video_id][question_id]["options"]) + "\n"
        # print(question)

        query = f"""Select the best answer to the following multiple-choice question based on the video. Respond with only the letter (A, B, C, or D) of the correct option. \n{question}The best answer is:"""
        messages = [
            {"role": "user", 
            "content": [
                    {
                        "type": "text",
                        "text": query
                    }
                ]
            },
        ]
        
        if image_path_list is not None:
            for image in image_path_list:
                with open(image, "rb") as image_file:
                    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
                image_message = {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    }
                }
                messages[0]["content"].append(image_message)
                
        output = ""
        response = call_gpt_with_messages(messages, "gpt_4_turbo")
        output += response.choices[0].message.content
    
        print("Query: ", query)
        print("Predict: ", output + "\n")

        save_file = os.path.join(save_path, f"{question_id}_pred.txt")
        with open(save_file, 'w') as f:
            f.write(output)


## MLVU

In [ ]:
for video_id in questions.keys():
    task = questions[video_id]["question_type"]
    save_path = f"{DATA_ROOT}/results/MLVU/gpt_4o_20/no_rag/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    # else:
    #     files = os.listdir(save_path)
    #     if len(files) == len(questions[video_id]["question"]):
    #         continue

    frames = os.listdir(f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}")
    frames.sort()
    n_frms = 20

    indices = np.linspace(0, len(frames), num=n_frms, endpoint=False, dtype=int).tolist()
    image_path_list = []
    for i in indices:
        image_path_list.append(f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}/{frames[i]}")

    # generate questions
    for i, q in enumerate(questions[video_id]["question"]):
        print(f"processing: {video_id} {q}")
        save_file = os.path.join(save_path, f"{q[:40]}_pred.txt")
        
        question = ""
        question += q + "\n"
        if "candidates" in questions[video_id] and len(questions[video_id]["candidates"]) != 0:
            candidates = questions[video_id]["candidates"][i]
            options = ['A', 'B', 'C', 'D']
            for i, c in enumerate(candidates):
                question += options[i] + ". " + c + "\n"

        if questions[video_id]["question_type"] not in ['8_sub_scene', '9_summary']:
            query = f"""Carefully watch this video and pay attention to every detail. Based on your observations, select the best option that accurately addresses the question. \n{question}Only choose the best option. Best option: ("""
            # query = f"""These frames are from a video. Please examine each frame in the sequence provided to understand the narrative or activities depicted. Based on your observations, select the option that best answer the question. \n{question}Only choose the best option. Best option: ("""
        else:
            query = f"""Carefully watch this video and pay attention to every detail. Based on your observations, answer the given questions. \n{question}"""
            # query = f"""These frames are from a video. Please examine each frame in the sequence provided to understand the narrative or activities depicted. Based on your observations, answer the given questions. \n{question}"""
        
        messages = [
            {"role": "user", 
            "content": [
                    {
                        "type": "text",
                        "text": query
                    }
                ]
            },
        ]
        
        if image_path_list is not None:
            for image in image_path_list:
                with open(image, "rb") as image_file:
                    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
                image_message = {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    }
                }
                messages[0]["content"].append(image_message)
                
        output = ""
        response = call_gpt_with_messages(messages, "gpt_4o")
        output += response.choices[0].message.content
    
        print("query: ", query)
        print("answer: ", output + "\n")

        with open(save_file, 'w') as f:
            f.write(output)


## Perception Test

In [ ]:
for video_id in questions_perception_test.keys():
    save_path = f"{DATA_ROOT}/results/Perception_Test/gpt_4o_10/no_rag/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    else:
        files = os.listdir(save_path)
        if len(files) == len(questions_perception_test[video_id]["question"]):
            continue

    frames = os.listdir(f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}")
    frames.sort()
    n_frms = 10

    indices = np.linspace(0, len(frames), num=n_frms, endpoint=False, dtype=int).tolist()
    image_path_list = []
    for i in indices:
        image_path_list.append(f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}/{frames[i]}")

    # generate questions
    for i in range(len(questions_perception_test[video_id]["question"])):
        question_id = questions_perception_test[video_id]["id"][i]
        print(f"processing: {video_id} {question_id}")

        question = ""
        question += questions_perception_test[video_id]["question"][i] + "\n"
        question += '\n'.join(questions_perception_test[video_id]["options"][i]) + "\n"
        # print(question)

        query = f"""Select the best answer to the following multiple-choice question based on the video. Respond with only the letter (A, B, C) of the correct option. \n{question}The best answer is:"""
        messages = [
            {"role": "user", 
            "content": [
                    {
                        "type": "text",
                        "text": query
                    }
                ]
            },
        ]
        
        if image_path_list is not None:
            for image in image_path_list:
                with open(image, "rb") as image_file:
                    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
                image_message = {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "low"
                    }
                }
                messages[0]["content"].append(image_message)
        
        print("query: ", query)
        print("image_path_list: ", image_path_list)
        output = ""
        response = call_gpt_with_messages(messages, "gpt_4o")
        output += response.choices[0].message.content
        
        print("answer: ", output + "\n")
        
        save_file = os.path.join(save_path, f"{question_id}_pred.txt")
        with open(save_file, 'w') as f:
            f.write(output)


# Test 2: No Fine tuning/Self-Supervised Fine-tuning/Grouped-Supervised Fine-tuning (GPT4o)

## Video-MME

In [ ]:
for video_id in picked:
    save_path = f"{DATA_ROOT}/results/gpt_4o_10/gc_rag_uni_srt/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    else:
        files = os.listdir(save_path)
        # if len(files) == 3:
        #     continue

    for question_id in questions[video_id]:
        print(f"processing: {video_id} {question_id}")
        save_file = os.path.join(save_path, f"{question_id}_pred.txt")
        # if os.path.exists(save_file):
        #     continue

        frames = os.listdir(f"{DATA_ROOT}/rag_adapter_sampled_frames/Video-MME/10_frames_gc/{video_id}/{question_id}")
        frames.sort()
        frames_path = [f"{DATA_ROOT}/rag_adapter_sampled_frames/Video-MME/10_frames_gc/{video_id}/{question_id}/{frame}" for frame in frames]
        # print(frames)

        frame_index = [int(frame.split(".")[0]) for frame in frames]
        frame_index = [frame - 2 if frame - 2 >= 0 else frame for frame in frame_index]
        frame_index.sort()
        timestamp = []
        for idx in frame_index:
            time = strftime("%H:%M:%S", gmtime(int(idx)))
            timestamp.append(time)
        
        srt_path = os.path.join(f"{DATA_ROOT}/dataset/Video-MME/subtitle", f"{video_id}.srt")
        subtitles = ""
        uni_sub = set()
        if os.path.exists(srt_path):
            subs = pysrt.open(srt_path)
            unique_timestamp = list(dict.fromkeys(timestamp))
            for _, times in enumerate(unique_timestamp):
                for sub in subs:  
                    start_time = sub.start.to_time().strftime('%H:%M:%S.%f')[:-3]
                    end_time = sub.end.to_time().strftime('%H:%M:%S.%f')[:-3]
                    content = re.sub(r'<[^>]+>', '', sub.text)
                    if times>str(start_time) and times < str(end_time):
                        if sub.start.to_time() in uni_sub:
                            continue
                        else:
                            uni_sub.add(sub.start.to_time())  
                            subtitles += f"From {start_time} to {end_time}: {content}\n"
        
        q_o = ""
        q_o += questions[video_id][question_id]["question"] + "\n"
        q_o += '\n'.join(questions[video_id][question_id]["options"]) + "\n"

        # with srt
        query = f"""This video’s subtitles are listed below:\n{subtitles}\nSelect the best answer to the following multiple-choice question based on the video. Respond with only the letter (A, B, C, or D) of the correct option. \n{q_o}The best answer is:"""
        # without srt
        # query = f"""Select the best answer to the following multiple-choice question based on the video. Respond with only the letter (A, B, C, or D) of the correct option. \n{q_o}The best answer is:"""
        
        messages = [
            {"role": "user", 
            "content": [
                    {
                        "type": "text",
                        "text": query
                    }
                ]
            },
        ]
        
        if frames_path is not None:
            for image in frames_path:
                with open(image, "rb") as image_file:
                    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
                image_message = {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    }
                }
                messages[0]["content"].append(image_message)
        output = ""
        response = call_gpt_with_messages(messages, "gpt_4o")
        output += response.choices[0].message.content

        print("query: ", query)
        print("answer: ", output + "\n")

        with open(save_file, 'w') as f:
            f.write(output)


## MLVU

In [ ]:
def mlvu_gpt_test(video_id):
    # for video_id in questions.keys():
    task = questions[video_id]["question_type"]
    save_path = f"{DATA_ROOT}/results/MLVU/gpt_4o_20/gc_rag/{video_id}/"

    if not os.path.exists(save_path):
        os.makedirs(save_path)
    # else:
    #     files = os.listdir(save_path)
    #     if len(files) == len(questions[video_id]["question"]):
    #         return 

    for i, q in enumerate(questions[video_id]["question"]):
        print(f"processing: {video_id} {q}")
        question_id = q[:40]
        save_file = os.path.join(save_path, f"{question_id}_pred.txt")
        # if os.path.exists(save_file):
        #     continue

        frames = os.listdir(f"{DATA_ROOT}/rag_adapter_sampled_frames/MLVU/20_frames_gc/{task}/{video_id}/{question_id}")
        frames.sort()
        frames_path = [f"{DATA_ROOT}/rag_adapter_sampled_frames/MLVU/20_frames_gc/{task}/{video_id}/{question_id}/{frame}" for frame in frames]
        # print(frames)

        frame_index = [int(frame.split(".")[0]) for frame in frames]
        frame_index = [frame - 2 if frame - 2 >= 0 else frame for frame in frame_index]
        frame_index.sort()
        
        question = ""
        question += q + "\n"
        if "candidates" in questions[video_id] and len(questions[video_id]["candidates"]) != 0:
            candidates = questions[video_id]["candidates"][i]
            options = ['A', 'B', 'C', 'D']
            for i, c in enumerate(candidates):
                question += options[i] + ". " + c + "\n"

        if questions[video_id]["question_type"] not in ['8_sub_scene', '9_summary']:
            query = f"""Carefully watch this video and pay attention to every detail. Based on your observations, select the best option that accurately addresses the question. \n{question}Only choose the best option. Best option: ("""
            # query = f"""These frames are from a video. Please examine each frame in the sequence provided to understand the narrative or activities depicted. Based on your observations, select the option that best answer the question. \n{question}Only choose the best option. Best option: ("""
        else:
            query = f"""Carefully watch this video and pay attention to every detail. Based on your observations, answer the given questions. \n{question}"""
            # query = f"""These frames are from a video. Please examine each frame in the sequence provided to understand the narrative or activities depicted. Based on your observations, answer the given questions. \n{question}"""
        
        messages = [
            {"role": "user", 
            "content": [
                    {
                        "type": "text",
                        "text": query
                    }
                ]
            },
        ]
        
        if frames_path is not None:
            for image in frames_path:
                with open(image, "rb") as image_file:
                    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
                image_message = {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    }
                }
                messages[0]["content"].append(image_message)
                
        output = ""
        response = call_gpt_with_messages(messages, "gpt_4o")
        output += response.choices[0].message.content

        print("query: ", query)
        print("answer: ", output + "\n")

        with open(save_file, 'w') as f:
            f.write(output)

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(mlvu_gpt_test, video_id)  for video_id in questions.keys()]
    for job in as_completed(futures):
        result = job.result(timeout=None)
        time.sleep(1)


## Perception Test

In [ ]:
for video_id in questions_perception_test.keys():
    save_path = f"{DATA_ROOT}/results/Perception_Test/gpt_4o_10/gc_rag/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    else:
        files = os.listdir(save_path)
        if len(files) == len(questions_perception_test[video_id]["question"]):
            continue

    for i in range(len(questions_perception_test[video_id]["question"])):
        question_id = questions_perception_test[video_id]["id"][i]
        print(f"processing: {video_id} {question_id}")

        save_file = os.path.join(save_path, f"{i}_pred.txt")
        # if os.path.exists(save_file):
        #     continue

        frames = os.listdir(f"{DATA_ROOT}/rag_adapter_sampled_frames/Perception_Test/10_frames_gc/{video_id}/{question_id}")
        frames.sort()
        frames_path = [f"{DATA_ROOT}/rag_adapter_sampled_frames/Perception_Test/10_frames_gc/{video_id}/{question_id}/{frame}" for frame in frames]
        # print(frames)

        frame_index = [int(frame.split(".")[0]) for frame in frames]
        frame_index = [frame - 2 if frame - 2 >= 0 else frame for frame in frame_index]
        frame_index.sort()
        
        q_o = ""
        q_o += questions_perception_test[video_id]["question"][i] + "\n"
        q_o += '\n'.join(questions_perception_test[video_id]["options"][i]) + "\n"

        # without srt
        query = f"""Select the best answer to the following multiple-choice question based on the video. Respond with only the letter (A, B, C) of the correct option. \n{q_o}The best answer is:"""
        
        messages = [
            {"role": "user", 
            "content": [
                    {
                        "type": "text",
                        "text": query
                    }
                ]
            },
        ]
        
        if frames_path is not None:
            for image in frames_path:
                with open(image, "rb") as image_file:
                    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
                image_message = {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "low"
                    }
                }
                messages[0]["content"].append(image_message)
        output = ""
        response = call_gpt_with_messages(messages, "gpt_4o")
        output += response.choices[0].message.content

        print("query: ", query)
        print("answer: ", output + "\n")

        with open(save_file, 'w') as f:
            f.write(output)


# Self-supervised contrast learning fine-tunes bge-m3 and clip:ViT-L/14

##  Implement a custom self-supervised contrast loss function

In [ ]:
"""Sentence Transformer Finetuning Engine."""
class SelfSupervisedFinetuneEngine(BaseEmbeddingFinetuneEngine):
    """Sentence Transformers Finetune Engine."""

    def __init__(
        self,
        dataset: EmbeddingQAFinetuneDataset,
        model_id: str = "BAAI/bge-small-en",
        model_output_path: str = "exp_finetune",
        batch_size: int = 16,
        val_dataset: Optional[EmbeddingQAFinetuneDataset] = None,
        loss: Optional[Any] = None,
        epochs: int = 2,
        show_progress_bar: bool = True,
        evaluation_steps: int = 50,
        use_all_docs: bool = False,
        trust_remote_code: bool = False,
        device: Optional[Any] = None,
    ) -> None:
        """Init params."""
        from sentence_transformers import InputExample, SentenceTransformer, losses
        from torch.utils.data import DataLoader

        self.dataset = dataset

        self.model_id = model_id
        self.model_output_path = model_output_path
        self.model = SentenceTransformer(
            model_id, trust_remote_code=trust_remote_code, device=device
        )

        self.use_all_docs = use_all_docs

        examples: Any = []
        for query_id, query in dataset.queries.items():
            if use_all_docs:
                for node_id in dataset.relevant_docs[query_id]:
                    text = dataset.corpus[node_id]
                    example = InputExample(texts=[query, text])
                    examples.append(example)
            else:
                node_id = dataset.relevant_docs[query_id][0]
                text = dataset.corpus[node_id]
                example = InputExample(texts=[query, text])
                examples.append(example)

        self.examples = examples

        self.loader: DataLoader = DataLoader(examples, batch_size=batch_size)

        # define evaluator
        from sentence_transformers.evaluation import InformationRetrievalEvaluator

        evaluator: Optional[InformationRetrievalEvaluator] = None
        if val_dataset is not None:
            evaluator = InformationRetrievalEvaluator(
                val_dataset.queries, val_dataset.corpus, val_dataset.relevant_docs
            )
        self.evaluator = evaluator

        # define loss
        self.loss = loss or losses.MultipleNegativesRankingLoss(self.model)

        self.epochs = epochs
        self.show_progress_bar = show_progress_bar
        self.evaluation_steps = evaluation_steps
        self.warmup_steps = int(len(self.loader) * epochs * 0.1)

    def finetune(self, **train_kwargs: Any) -> None:
        """Finetune model."""
        self.model.fit(
            train_objectives=[(self.loader, self.loss)],
            epochs=self.epochs,
            warmup_steps=self.warmup_steps,
            output_path=self.model_output_path,
            show_progress_bar=self.show_progress_bar,
            evaluator=self.evaluator,
            evaluation_steps=self.evaluation_steps,
            scheduler="WarmupCosine"
        )

    def get_finetuned_model(self, **model_kwargs: Any) -> BaseEmbedding:
        """Gets finetuned model."""
        embed_model_str = "local:" + self.model_output_path
        return resolve_embed_model(embed_model_str)


# Self-supervised loss function
class MultipleNegativesRankingLoss(nn.Module):
    def __init__(self, model: SentenceTransformer, scale: float = 20.0, similarity_fct=util.cos_sim) -> None:
        super().__init__()
        self.model = model
        self.scale = scale
        self.similarity_fct = similarity_fct
        self.cross_entropy_loss = nn.CrossEntropyLoss()

    def forward(self, sentence_features: Iterable[dict[str, Tensor]], labels: Tensor) -> Tensor:
        reps = [self.model(sentence_feature)["sentence_embedding"] for sentence_feature in sentence_features]
        # length = 2 (archor, positive) or 3 (archor, positive, negative)
        # print(f"len(reps): {len(reps)}")
        embeddings_a = reps[0]
        embeddings_b = torch.cat(reps[1:])
        # embeddings_a.shape: torch.Size([10, 1024]) (bs, dim)
        # embeddings_b.shape: torch.Size([10, 1024]) (bs, dim)
        # print(f"embeddings_a.shape: {embeddings_a.shape}")
        # print(f"embeddings_b.shape: {embeddings_b.shape}")

        scores = self.similarity_fct(embeddings_a, embeddings_b) * self.scale
        # Example a[i] should match with b[i]
        range_labels = torch.arange(0, scores.size(0), device=scores.device)
        return self.cross_entropy_loss(scores, range_labels)

    def get_config_dict(self) -> dict[str, Any]:
        return {"scale": self.scale, "similarity_fct": self.similarity_fct.__name__}

    @property
    def citation(self) -> str:
        return """
@misc{henderson2017efficient,
    title={Efficient Natural Language Response Suggestion for Smart Reply},
    author={Matthew Henderson and Rami Al-Rfou and Brian Strope and Yun-hsuan Sung and Laszlo Lukacs and Ruiqi Guo and Sanjiv Kumar and Balint Miklos and Ray Kurzweil},
    year={2017},
    eprint={1705.00652},
    archivePrefix={arXiv},
    primaryClass={cs.CL}
}
"""


## Finetune BGE-M3

In [ ]:
train_dataset = EmbeddingQAFinetuneDataset.from_json(f"{WORK_ROOT}/finetune/video_mme/train_dataset.json")
val_dataset = EmbeddingQAFinetuneDataset.from_json(f"{WORK_ROOT}/finetune/video_mme/test_dataset.json")

finetune_engine_sc = SelfSupervisedFinetuneEngine(
    train_dataset,
    model_id=f"{BGE_MODEL}",
    model_output_path=f"{WORK_ROOT}/finetune/model/bge_m3_finetuned_sc",
    batch_size=16,
    val_dataset=val_dataset,
    epochs=2,
    evaluation_steps=5000
)
finetune_engine_sc.loss = MultipleNegativesRankingLoss(finetune_engine_sc.model)


In [ ]:
# Start fine-tuning
finetune_engine_sc.finetune()


## Fine tune CLIP model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_processor = clip.load("ViT-L/14", device=device)


In [ ]:
# Prepare custom text and image data
finetune_datasets = ['MSVD-QA','MSRVTT-QA','ActivityNet-QA','TGIF-QA']

train_querys = []
test_querys = []
train_image_paths = []
test_image_paths = []
for dataset in finetune_datasets:
    iformat = "jpg" if dataset != "TGIF-QA" else "png"
    for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/{dataset}/captions"):
        if len(ds) == 0:
            video_id = os.path.basename(root)
            frames_num = os.listdir(os.path.join(f"{DATA_ROOT}/dataset/{dataset}/frames", video_id))
            if len(frames_num) > 300:
                continue
            if len(fs) != 0:
                for f in fs:
                    image_index = f.split(".")[0]
                    image_path = f"{DATA_ROOT}/dataset/{dataset}/frames/{video_id}/{image_index}.{iformat}"
                    if video_id in train_questions_dict:
                        questions = train_questions_dict[video_id]
                        for question in questions:
                            train_querys.append(question)
                            train_image_paths.append(image_path)
                    if video_id in test_questions_dict:
                        questions = test_questions_dict[video_id]
                        for question in questions:
                            test_querys.append(question)
                            test_image_paths.append(image_path)
    # images = [Image.open(image_path) for image_path in image_paths]
# print(len(train_querys))            
# print(len(test_querys))
                                


In [ ]:
# Delete queries longer than 77
def check_query_length(queries, images):
    for i, query in enumerate(queries):
        try:
            q = clip.tokenize(query)
        except:
            queries.pop(i)
            images.pop(i)

check_query_length(train_querys, train_image_paths)
check_query_length(test_querys, test_image_paths)


In [ ]:
# Custom data sets
class ClipFintuneDataset(Dataset):
    def __init__(self, querys, image_paths, processor):
        self.querys = clip.tokenize(querys)        
        self.image_paths = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.querys)
    
    def __getitem__(self, idx):
        query = self.querys[idx]
        image = self.processor(Image.open(self.image_paths[idx]))
        return query, image

train_dataset = ClipFintuneDataset(train_querys, train_image_paths, clip_processor)
test_dataset = ClipFintuneDataset(test_querys, test_image_paths, clip_processor)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=True)


In [ ]:
# Define clip self-supervised contrastive loss function
def self_contrastive_loss(text_features, image_features, temperature=0.05):
    ## text_features: (bs, dim) ( , 768)
    ## image_features: (bs, dim) ( , 768)
    text_embeddings = F.normalize(text_features, p=2, dim=1)
    image_embeddings = F.normalize(image_features, p=2, dim=1)
    # print(text_embeddings.shape)
    # print(image_embeddings.shape)

    # Calculate the similarity between text and image
    logits = torch.mm(text_embeddings, image_embeddings.transpose(0, 1)) / temperature
    
    labels = torch.arange(len(text_embeddings)).to(text_embeddings.device) # [0, 1, 2, ..., bs-1]
    # print(labels)

    loss = F.cross_entropy(logits, labels)

    return loss


In [ ]:
# clip fine tuning
## optimizer
epoches = 2
global_step = 0
total_steps = len(train_dataloader) * epoches
warmup_steps = int(0.1 * total_steps)
optimizer = torch.optim.AdamW(clip_model.parameters(), lr=1e-5, betas=(0.9, 0.98), eps=1e-6 , weight_decay=0.02)
# scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
best_val_loss = float("inf")

# Start training
total_val_loss = 0.0
def eval_model(model, test_dataloader, step):
    model.eval()
    total_val_loss = 0.0
    val_p = tqdm(test_dataloader, desc=f"Validation at Iteration: {step}", total=len(test_dataloader), leave=False)
    with torch.no_grad():
        for v_b in val_p:
            q_v, i_v = v_b
            q_v = q_v.to(device)
            i_v = i_v.to(device)

            tf_v = clip_model.encode_text(q_v)
            if_v = clip_model.encode_image(i_v)

            val_loss = self_contrastive_loss(tf_v, if_v)
            total_val_loss += val_loss.item()
    
    avg_val_loss = total_val_loss / len(test_dataloader)

    print(f"Iteration {step}, Validation Loss: {avg_val_loss:.4f}")
    global best_val_loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(clip_model.state_dict(), f"{WORK_ROOT}/finetune/model/clip_best_finetuned_sc.pth")
        print(f"Best clip model saved at iteration {step} with Validation Loss: {best_val_loss:.4f}")
    # print("best_val_loss: ", best_val_loss)
    return

for param in clip_model.parameters():
    param.requires_grad = True  # Unfreeze all parameters for update

total_loss = 0.0
for epoch in range(epoches):
    
    clip_model.train()
    progress = tqdm(train_dataloader, desc=f"Training Epoch: {epoch+1}/{epoches}", total=len(train_dataloader), leave=False)

    for batch in progress:
        global_step += 1

        query, image = batch   
        query = query.to(device)
        image = image.to(device)

        # text_features, image_features = clip_model(query, image)
        text_features = clip_model.encode_text(query)
        image_features = clip_model.encode_image(image)

        cur_loss = self_contrastive_loss(text_features, image_features)

        optimizer.zero_grad()
        cur_loss.backward()

        # print out the gradients
        for name, param in clip_model.named_parameters():
            if param.grad is not None:
                print(f"Gradient of {name}: {param.grad.norm()}")

        optimizer.step()
        scheduler.step()

        total_loss += cur_loss.item()
        progress.set_postfix(loss=cur_loss.item())
        if global_step % 1000 == 0:
            cur_avg_loss = total_loss / global_step
            print(f"Epoch [{epoch+1}/{epoches}], Step [{global_step}/{total_steps}], Training Loss: {cur_avg_loss:.4f}")

        if global_step % 5000 == 0:
            eval_model(clip_model, test_dataloader, global_step)
            clip_model.train()

    average_loss = total_loss / global_step
    print(f"Epoch [{epoch+1}/{epoches}], Training Loss: {average_loss:.4f}")

eval_model(clip_model, test_dataloader, global_step)


# Grouped-supervised contrastive learning fine-tune bge-m3 and clip/L-14

## Implement a custom grouped-supervised contrastive loss function

In [ ]:
# Custom Fine-Tuning Function

"""Sentence Transformer Finetuning Engine."""
class GroupTransformersFinetuneEngine(BaseEmbeddingFinetuneEngine):
    """Sentence Transformers Finetune Engine."""

    def __init__(
        self,
        dataset: EmbeddingQAFinetuneDataset,
        model_id: str = "BAAI/bge-small-en",
        model_output_path: str = "exp_finetune",
        batch_size: int = 16,
        val_dataset: Optional[EmbeddingQAFinetuneDataset] = None,
        loss: Optional[Any] = None,
        epochs: int = 2,
        show_progress_bar: bool = True,
        evaluation_steps: int = 50,
        use_all_docs: bool = False,
        trust_remote_code: bool = False,
        device: Optional[Any] = None,
        train_hash_map: Optional[Dict[str, int]] = None,
    ) -> None:
        """Init params."""
        from sentence_transformers import InputExample, SentenceTransformer, losses
        from torch.utils.data import DataLoader

        self.dataset = dataset

        self.model_id = model_id
        self.model_output_path = model_output_path
        self.model = SentenceTransformer(
            model_id, trust_remote_code=trust_remote_code, device=device
        )

        self.use_all_docs = use_all_docs

        examples: Any = []
        for query_id, query in dataset.queries.items():
            if use_all_docs:
                for node_id in dataset.relevant_docs[query_id]:
                    text = dataset.corpus[node_id]
                    label = train_hash_map[node_id]
                    example = InputExample(texts=[query, text], label=label)
                    examples.append(example)
            else:
                node_id = dataset.relevant_docs[query_id][0]
                text = dataset.corpus[node_id]
                # add unique label for each node_id
                label = train_hash_map[node_id]
                example = InputExample(texts=[query, text], label=label)
                examples.append(example)

        self.examples = examples

        self.loader: DataLoader = DataLoader(examples, batch_size=batch_size)

        # define evaluator
        from sentence_transformers.evaluation import InformationRetrievalEvaluator

        evaluator: Optional[InformationRetrievalEvaluator] = None
        if val_dataset is not None:
            evaluator = InformationRetrievalEvaluator(
                val_dataset.queries, val_dataset.corpus, val_dataset.relevant_docs
            )
        self.evaluator = evaluator

        # define loss
        self.loss = loss or losses.MultipleNegativesRankingLoss(self.model)

        self.epochs = epochs
        self.show_progress_bar = show_progress_bar
        self.evaluation_steps = evaluation_steps
        self.warmup_steps = int(len(self.loader) * epochs * 0.1)

    def finetune(self, **train_kwargs: Any) -> None:
        """Finetune model."""
        self.model.fit(
            train_objectives=[(self.loader, self.loss)],
            epochs=self.epochs,
            warmup_steps=self.warmup_steps,
            output_path=self.model_output_path,
            show_progress_bar=self.show_progress_bar,
            evaluator=self.evaluator,
            evaluation_steps=self.evaluation_steps,
            scheduler="WarmupCosine"
        )

    def get_finetuned_model(self, **model_kwargs: Any) -> BaseEmbedding:
        """Gets finetuned model."""
        embed_model_str = "local:" + self.model_output_path
        return resolve_embed_model(embed_model_str)

# Grouped-supervised Contrastive Loss Function
class GroupSupervisedContrastiveLoss(nn.Module):
    def __init__(self, model: SentenceTransformer, scale: float = 20.0, similarity_fct=util.cos_sim) -> None:
        """
        """
        super().__init__()
        self.model = model
        self.scale = scale
        self.similarity_fct = similarity_fct
        self.cross_entropy_loss = nn.CrossEntropyLoss()

    def forward(self, sentence_features: Iterable[dict[str, Tensor]], labels: Tensor) -> Tensor:
        reps = [self.model(sentence_feature)["sentence_embedding"] for sentence_feature in sentence_features]
        # length = 2 (archor, positive) or 3 (archor, positive, negative)
        # print(f"len(reps): {len(reps)}")

        embeddings_a = reps[0]
        embeddings_b = torch.cat(reps[1:])

        # embeddings_a.shape: torch.Size([10, 1024]) (bs, dim)
        # embeddings_b.shape: torch.Size([10, 1024]) (bs, dim)
        # print(f"embeddings_a.shape: {embeddings_a.shape}")
        # print(f"embeddings_b.shape: {embeddings_b.shape}")

        similarities = self.similarity_fct(embeddings_a, embeddings_b) * self.scale

        # # Create a Mask Matrix for Equal Labels
        # labels = labels.unsqueeze(1)
        # positive_mask = torch.eq(labels, labels.T).float().to(labels.device)

        # # Calculate logarithmic probability
        # exp_logits = torch.exp(logits)
        # log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

        # # Calculate the loss for each sample
        # mean_log_prob_pos = (positive_mask * log_prob).sum(dim=1) / positive_mask.sum(dim=1)
        # loss = -mean_log_prob_pos.mean()

        # Create a Mask Matrix for Equal Labels
        labels = labels.unsqueeze(1) # (bs, 1)

        # Construct target labels: if there is only one label of the same class, the target is itself. If there are multiple labels of the same class, create a target label for each sample.
        # targets = torch.arange(labels.shape[0], device=labels.device)
        all_possible_index = []
        for i in range(labels.shape[0]):
            matching_indices = (labels == labels[i]).nonzero(as_tuple=True)[0]
            # Indices of all samples with the same label as the current one.
            all_possible_index.append(matching_indices)
            # All samples with different labels from the current one.
            # possible_indices = matching_indices[matching_indices != i]
            # if len(possible_indices) > 0:
            #     all_possible_index.append(possible_indices)
            #     # selected_index = possible_indices[torch.randint(len(possible_indices), (1,))]
            # else:
            #     all_possible_index.append([i])
            #     # selected_index = i
            # # targets[i] = selected_index
                
        all_combinations = list(itertools.product(*all_possible_index))
        all_targets_combinations = [torch.tensor(list(combination), device=labels.device) for combination in all_combinations]

        # logits = similarities.clone()
        # for i in range(logits.shape[0]):
        #     # if sum(logits[i][(labels.squeeze() == labels[i].item()) & (torch.arange(labels.shape[0], device=labels.device) != i)]) > 0:
        #     #     print("Warning: Same Label: ", i, "and: ", labels)
        #     logits[i][(labels.squeeze() == labels[i].item()) & (torch.arange(labels.shape[0], device=labels.device) != i)] = float(-50)

        loss = 0.0 
        # print(logits)
        # print("all_targets_combinations: ", all_targets_combinations)
        for targets in all_targets_combinations:
            # Mask samples of the same class based on the targets
            logits = similarities.clone()
            for i in range(logits.shape[0]):
                logits[i][(labels.squeeze() == labels[i].item()) & (torch.arange(labels.shape[0], device=labels.device) != targets[i])] = float(-50)
            loss += F.cross_entropy(logits, targets)
        # print("total loss: ", loss)
        loss = loss / len(all_targets_combinations)

        return loss

    def get_config_dict(self) -> dict[str, Any]:
        return {"scale": self.scale, "similarity_fct": self.similarity_fct.__name__}


## Fine tune BGE-M3

In [ ]:
train_dataset = EmbeddingQAFinetuneDataset.from_json(f"{WORK_ROOT}/finetune/video_mme/train_dataset.json")
val_dataset = EmbeddingQAFinetuneDataset.from_json(f"{WORK_ROOT}/finetune/video_mme/test_dataset.json")

# Gets a unique label for each node_id
train_hash_map = {}
train_label = 0

def string_to_unique_int(s):
    global train_label
    if s not in train_hash_map:
        train_hash_map[s] = train_label
        train_label += 1
    return train_hash_map[s]

for query_id, query in train_dataset.queries.items():
    node_id = train_dataset.relevant_docs[query_id][0]
    string_to_unique_int(node_id)


finetune_engine_gc = GroupTransformersFinetuneEngine(
    train_dataset,
    model_id=f"{BGE_MODEL}",
    model_output_path=f"{WORK_ROOT}/finetune/model/bge_m3_finetuned_gc",
    batch_size=32,
    val_dataset=val_dataset,
    epochs=2,
    evaluation_steps=1000,
    train_hash_map=train_hash_map
)
finetune_engine_gc.loss = GroupSupervisedContrastiveLoss(finetune_engine_gc.model)


In [ ]:
# Start fine-tuning
finetune_engine_gc.finetune()


## Fine tune CLIP model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_processor = clip.load("ViT-L/14", device=device)


In [ ]:
# Prepare custom text and image data

finetune_datasets = ['MSVD-QA','MSRVTT-QA','ActivityNet-QA','TGIF-QA']

train_querys = []
test_querys = []
train_image_paths = []
test_image_paths = []

# Sample only the 50th percentile
# for dataset in finetune_datasets:
#     iformat = "jpg" if dataset != "TGIF-QA" else "png"
#     for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/{dataset}/captions"):
#         if len(ds) == 0:
#             video_id = os.path.basename(root)
#             frames_num = os.listdir(os.path.join(f"{DATA_ROOT}/dataset/{dataset}/frames", video_id))
#             # discard videos with more than 300 frames
#             if len(frames_num) > 300:
#                 continue
#             if len(fs) != 0:
#                 for f in fs:
#                     image_index = f.split(".")[0]
#                     image_path = f"{DATA_ROOT}/dataset/{dataset}/frames/{video_id}/{image_index}.{iformat}"
#                     if video_id in train_questions_dict:
#                         questions = train_questions_dict[video_id]
#                         for question in questions:
#                             train_querys.append(question)
#                             train_image_paths.append(image_path)
#                     if video_id in test_questions_dict:
#                         questions = test_questions_dict[video_id]
#                         for question in questions:
#                             test_querys.append(question)
#                             test_image_paths.append(image_path)

# Sample 25,50, and 75th percentiles, or all if the number of frames is only 1 to 3
for dataset in finetune_datasets:
    for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/{dataset}/frames"):
        if len(ds) == 0:
            image_names = []
            video_id = os.path.basename(root)
            frames = os.listdir(root)
            frames.sort()
            frames_idx = list(range(0, len(frames)))
            if len(frames) <= 3:
                image_names = frames
            else:
                q1 = int(np.percentile(frames_idx, 25))
                q2 = int(np.percentile(frames_idx, 50))
                q3 = int(np.percentile(frames_idx, 75))
                image_names.append(frames[q1])
                image_names.append(frames[q2])
                image_names.append(frames[q3])

            # discard videos with more than 300 frames
            if len(frames) > 300:
                continue
            if len(fs) != 0:
                if video_id in train_questions_dict:
                    questions = train_questions_dict[video_id]
                    for question in questions:
                        image_index = np.random.randint(0, len(image_names))
                        image_path = os.path.join(root, image_names[image_index])
                        train_querys.append(question)
                        train_image_paths.append(image_path)
                if video_id in test_questions_dict:
                    questions = test_questions_dict[video_id]
                    for question in questions:
                        image_index = np.random.randint(0, len(image_names))
                        image_path = os.path.join(root, image_names[image_index])
                        test_querys.append(question)
                        test_image_paths.append(image_path)

# print(len(train_querys))            
# print(len(test_querys))
                                


In [ ]:
# Delete queries longer than 77
def check_query_length(queries, images):
    for i, query in enumerate(queries):
        try:
            q = clip.tokenize(query)
        except:
            queries.pop(i)
            images.pop(i)

check_query_length(train_querys, train_image_paths)
check_query_length(test_querys, test_image_paths)


In [ ]:
# Gets a unique label for each node_id
train_hash_map = {}
train_label = 0
test_hash_map = {}
test_label = 0

def generate_train_labels(s):
    global train_label
    if s not in train_hash_map:
        train_hash_map[s] = train_label
        train_label += 1
    return train_hash_map[s]

def generate_test_labels(s):
    global test_label
    if s not in test_hash_map:
        test_hash_map[s] = test_label
        test_label += 1
    return test_hash_map[s]

for path in train_image_paths:
    generate_train_labels(path)
for path in test_image_paths:
    generate_test_labels(path)  

# Custom data sets
class ClipFintuneDataset(Dataset):
    def __init__(self, querys, image_paths, processor, hash_map):
        self.querys = clip.tokenize(querys)   
        self.image_paths = image_paths
        self.processor = processor
        self.hash_map = hash_map
        # self.org_querys = querys

    def __len__(self):
        return len(self.querys)
    
    def __getitem__(self, idx):
        query = self.querys[idx]
        image = self.processor(Image.open(self.image_paths[idx]))
        label = self.hash_map[self.image_paths[idx]]
        return query, image, label

train_dataset = ClipFintuneDataset(train_querys, train_image_paths, clip_processor, train_hash_map)
test_dataset = ClipFintuneDataset(test_querys, test_image_paths, clip_processor, test_hash_map)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

print(len(train_dataloader))


In [ ]:
# Define the clip grouped-supervised contrastive loss function
def group_supervised_contrastive__loss(text_features, image_features, labels, temperature=0.05):
    ## text_features: (bs, dim) ( , 768)
    ## image_features: (bs, dim) ( , 768)
    # if torch.isnan(text_features).any() or torch.isnan(image_features).any():
    #     raise ValueError("NaN values found in pre-normalized features.")
    text_embeddings = F.normalize(text_features, p=2, dim=1)
    image_embeddings = F.normalize(image_features, p=2, dim=1)
    # if torch.isnan(text_embeddings).any() or torch.isnan(image_embeddings).any():
    #     print("NaN values found in normalized features.")
        # return -1 
    # print("text shape: ", text_embeddings.shape)
    # print("image shape: ", image_embeddings.shape)
    
    similarities = torch.mm(text_embeddings, image_embeddings.transpose(0, 1)) / temperature

    # similarities = similarities - similarities.max(dim=1, keepdim=True).values

    # exp_logits = torch.exp(similarities)
    # log_prob = similarities - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

    # mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-12)
    # if torch.all(mean_log_prob_pos == 0):
    #     print("Warning: All mean_log_prob_pos are zero. Skipping this batch.")
    #     return -1
    
    # loss = -mean_log_prob_pos.mean()

    # Create a Mask Matrix for Equal Labels
    labels = labels.unsqueeze(1) # (bs, 1)
    # print("labels: ", labels)
    # mask = torch.eq(labels, labels.T).float().to(labels.device)
    # if mask.sum() == 0:
    #     print("Warning: No positive pairs in the batch. Skipping this batch.")
    #     return -1 
    
    # Construct target labels: if there is only one label of the same class, the target is itself. If there are multiple labels of the same class, create a target label for each sample.
    # targets = torch.arange(labels.shape[0], device=labels.device)
    all_possible_index = []
    for i in range(labels.shape[0]):
        matching_indices = (labels == labels[i]).nonzero(as_tuple=True)[0]
        # Indices of all samples with the same label as the current one.
        all_possible_index.append(matching_indices)
        # # All samples with different labels from the current one.
        # possible_indices = matching_indices[matching_indices != i]
        # if len(possible_indices) > 0:
        #     all_possible_index.append(possible_indices)
        #     # selected_index = possible_indices[torch.randint(len(possible_indices), (1,))]
        # else:
        #     all_possible_index.append([i])
        #     # selected_index = i
        # # targets[i] = selected_index

    all_combinations = list(itertools.product(*all_possible_index))

    all_targets_combinations = [torch.tensor(list(combination), device=labels.device) for combination in all_combinations]

    # logits = similarities.clone()
    # for i in range(logits.shape[0]):
    #     # if sum(logits[i][(labels.squeeze() == labels[i].item()) & (torch.arange(labels.shape[0], device=labels.device) != i)]) > 0:
    #     #     print("Warning: Same Label: ", i, "and: ", labels)
    #     logits[i][(labels.squeeze() == labels[i].item()) & (torch.arange(labels.shape[0], device=labels.device) != i)] = float(-50)

    loss = 0.0 
    # print(logits)
    # print("all_targets_combinations: ", all_targets_combinations)
    for targets in all_targets_combinations:
        logits = similarities.clone()
        for i in range(logits.shape[0]):
            logits[i][(labels.squeeze() == labels[i].item()) & (torch.arange(labels.shape[0], device=labels.device) != targets[i])] = float(-50)

        loss += F.cross_entropy(logits, targets)
    # print("total loss: ", loss)
    loss = loss / len(all_targets_combinations)
    return loss


In [ ]:
epoches = 2
global_step = 0
total_steps = len(train_dataloader) * epoches
warmup_steps = int(0.1 * total_steps)
optimizer = torch.optim.AdamW(clip_model.parameters(), lr=1e-5, betas=(0.9, 0.98), eps=1e-6 , weight_decay=0.01)
# scheduler = lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
best_val_loss = float("inf")

# Start training
total_val_loss = 0.0
def eval_model(model, test_dataloader, step):
    model.eval()
    total_val_loss = 0.0
    val_p = tqdm(test_dataloader, desc=f"Validation at Iteration: {step}", total=len(test_dataloader), leave=False)
    with torch.no_grad():
        for v_b in val_p:
            q_v, i_v, label = v_b
            q_v = q_v.to(device)
            i_v = i_v.to(device)
            label = label.to(device)
            tf_v = clip_model.encode_text(q_v)
            if_v = clip_model.encode_image(i_v)
            val_loss = group_supervised_contrastive__loss(tf_v, if_v, label)
            total_val_loss += val_loss.item()
    
    avg_val_loss = total_val_loss / len(test_dataloader)

    print(f"Iteration {step}, Validation Loss: {avg_val_loss:.4f}")
    global best_val_loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(clip_model.state_dict(), f"{WORK_ROOT}/finetune/model/clip_best_finetuned_gc.pth")
        print(f"Best clip model saved at iteration {step} with Validation Loss: {best_val_loss:.4f}")
    return

for param in clip_model.parameters():
    param.requires_grad = True

total_loss = 0.0
for epoch in range(epoches):
    
    clip_model.train()
    progress = tqdm(train_dataloader, desc=f"Training Epoch: {epoch+1}/{epoches}", total=len(train_dataloader), leave=False)

    for batch in progress:
        global_step += 1

        # query, image, label = batch   
        query, image, label = batch   
        query = query.to(device)
        image = image.to(device)
        label = label.to(device)
        text_features = clip_model.encode_text(query)
        image_features = clip_model.encode_image(image)
        cur_loss = group_supervised_contrastive__loss(text_features, image_features, label)
   
        if torch.isnan(cur_loss) or torch.isinf(cur_loss):
            print("Warning: Loss is NaN or inf, skipping this step.")
            continue
        optimizer.zero_grad()
        cur_loss.backward()

        # print out the gradients
        # for name, param in clip_model.named_parameters():
        #     if param.grad is not None:
        #         print(f"Gradient of {name}: {param.grad.norm()}")

        optimizer.step()
        scheduler.step()

        total_loss += cur_loss.item()
        # print("accum loss: ", total_loss)
        progress.set_postfix(loss=cur_loss.item())
        if global_step % 100 == 0:
            cur_avg_loss = total_loss / global_step
            print(f"Epoch [{epoch+1}/{epoches}], Step [{global_step}/{total_steps}], Training Loss: {cur_avg_loss:.4f}")

        if global_step % 1000 == 0:
            eval_model(clip_model, test_dataloader, global_step)
            clip_model.train()

    average_loss = total_loss / global_step
    print(f"Epoch [{epoch+1}/{epoches}], Training Loss: {average_loss:.4f}")

eval_model(clip_model, test_dataloader, global_step)


# Load the fine-tuned model

In [ ]:
# bge_m3 = HuggingFaceEmbedding(model_name="{BGE_MODEL}",device="cuda")
# clip = ClipEmbedding(model_name="ViT-L/14")

# bge_m3_sc = finetune_engine_sc.get_finetuned_model()
# clip_sc = ClipEmbedding(model_name="ViT-L/14", ck_path="{WORK_ROOT}/finetune/model/clip_best_finetuned_sc.pth")

bge_m3_gc = finetune_engine_gc.get_finetuned_model()
clip_gc = ClipEmbedding(model_name="ViT-L/14", ck_path=f"{WORK_ROOT}/finetune/model/clip_best_finetuned_gc.pth")


# Visualize search results

In [ ]:
def plot_images(image_paths):
    images_shown = 0
    plt.figure(figsize=(16, 9))
    for img_path in image_paths:
        if os.path.isfile(img_path):
            image = Image.open(img_path)

            plt.subplot(4, 4, images_shown + 1)
            plt.imshow(image)
            plt.xticks([])
            plt.yticks([])

            images_shown += 1
            if images_shown >= 16:
                break


In [ ]:
video_id = "4H8hcvNeWtg"
question = "What is the total number of people in the video?"
documents = SimpleDirectoryReader(f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}").load_data()
documents.extend(SimpleDirectoryReader(f"{DATA_ROOT}/dataset/Video-MME/captions/{video_id}").load_data())

# no_ft
client_no_ft = qdrant_client.QdrantClient(path = f"./qdrant_db/frames/no_ft/{video_id}")
text_store_no_ft = QdrantVectorStore(
    client=client_no_ft, collection_name="text_collection"
)
image_store_no_ft = QdrantVectorStore(
    client=client_no_ft, collection_name="image_collection"
)
storage_context_no_ft = StorageContext.from_defaults(
    vector_store=text_store_no_ft, image_store=image_store_no_ft
)
index_no_ft = MultiModalVectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context_no_ft,
    embed_model=bge_m3,
    image_embed_model=clip
)

# sc
client_sc = qdrant_client.QdrantClient(path = f"./qdrant_db/frames/sc/{video_id}")
text_store_sc = QdrantVectorStore(
    client=client_sc, collection_name="text_collection"
)
image_store_sc = QdrantVectorStore(
    client=client_sc, collection_name="image_collection"
)
storage_context_sc = StorageContext.from_defaults(
    vector_store=text_store_sc, image_store=image_store_sc
)
index_sc = MultiModalVectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context_sc,
    embed_model=bge_m3_sc,
    image_embed_model=clip_sc
)

# gc
client_gc = qdrant_client.QdrantClient(path = f"./qdrant_db/frames/gc/{video_id}")
text_store_gc = QdrantVectorStore(
    client=client_gc, collection_name="text_collection"
)
image_store_gc = QdrantVectorStore(
    client=client_gc, collection_name="image_collection"
)
storage_context_gc = StorageContext.from_defaults(
    vector_store=text_store_gc, image_store=image_store_gc
)
index_gc = MultiModalVectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context_gc,
    embed_model=bge_m3_gc,
    image_embed_model=clip_gc
)

index_no_ft.storage_context.persist(persist_dir=f"./qdrant_db/frames_storage/no_ft/{video_id}")
retriever_no_ft = index_no_ft.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
retrieval_results_no_ft = retriever_no_ft.retrieve(question)

index_sc.storage_context.persist(persist_dir=f"./qdrant_db/frames_storage/sc/{video_id}")
retriever_sc = index_sc.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
retrieval_results_sc = retriever_sc.retrieve(question)

index_gc.storage_context.persist(persist_dir=f"./qdrant_db/frames_storage/gc/{video_id}")
retriever_gc = index_gc.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
retrieval_results_gc = retriever_gc.retrieve(question)


In [ ]:
image_scores_no_ft = defaultdict(float)
image_scores_sc = defaultdict(float)
image_scores_gc = defaultdict(float)

# no_ft
for res_node in retrieval_results_no_ft:
    if isinstance(res_node.node, ImageNode):
        image_index = res_node.node.metadata['file_name'].split(".")[0]
        image_scores_no_ft[res_node.node.metadata["file_path"]] += res_node.get_score()
    else:
        image_index = res_node.node.metadata['file_name'].split(".")[0]
        video_id = res_node.node.metadata["file_path"].split("/")[-2]
        image_scores_no_ft[f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()

selected_items_no_ft = mmr_selection(image_scores_no_ft, bge_m3, clip, lambda_param=0.7, top_k=10)

# sc
for res_node in retrieval_results_sc:
    if isinstance(res_node.node, ImageNode):
        image_index = res_node.node.metadata['file_name'].split(".")[0]
        image_scores_sc[res_node.node.metadata["file_path"]] += res_node.get_score()
    else:
        image_index = res_node.node.metadata['file_name'].split(".")[0]
        video_id = res_node.node.metadata["file_path"].split("/")[-2]
        image_scores_sc[f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()

selected_items_sc = mmr_selection(image_scores_sc, bge_m3_sc, clip_sc, lambda_param=0.7, top_k=10)

# gc
for res_node in retrieval_results_gc:
    if isinstance(res_node.node, ImageNode):
        image_index = res_node.node.metadata['file_name'].split(".")[0]
        image_scores_gc[res_node.node.metadata["file_path"]] += res_node.get_score()
    else:
        image_index = res_node.node.metadata['file_name'].split(".")[0]
        video_id = res_node.node.metadata["file_path"].split("/")[-2]
        image_scores_gc[f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
selected_items_gc = mmr_selection(image_scores_gc, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=10)
# sort = sorted(image_scores_gc.items(), key=lambda x: x[1], reverse=True)
# print(sort)
# plot_images(selected_items_gc)


In [ ]:
video_id = "4H8hcvNeWtg"
question = "What is the total number of people in the video?"

frames = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}")
frames.sort()
n_frms = 10

# All video frames
all_frames_path = [f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{frame}" for frame in frames]
# Uniformly sampled video frames
uniform_indices = np.linspace(0, len(frames), num=n_frms, endpoint=False, dtype=int).tolist()
# RAG-Adapter Sampled frame's index
rag_indices_no_ft = []
for rag_path in selected_items_no_ft:
    for i, path in enumerate(all_frames_path):
        if rag_path == path:
            rag_indices_no_ft.append(i)
rag_indices_sc = []
for rag_path in selected_items_sc:
    for i, path in enumerate(all_frames_path):
        if rag_path == path:
            rag_indices_sc.append(i)
rag_indices_gc = []
for rag_path in selected_items_gc:
    for i, path in enumerate(all_frames_path):
        if rag_path == path:
            rag_indices_gc.append(i)

all_frames_embeddings = [torch.tensor(clip_gc._get_image_embedding(image), device='cpu') for image in all_frames_path]
text_embeddings = [torch.tensor(clip_gc._get_text_embedding(question), device='cpu')]

reducer = umap.UMAP(random_state=0, transform_seed=0)
text_embeddings_2d = reducer.fit_transform(text_embeddings)
all_frames_embeddings_2d = reducer.fit_transform(all_frames_embeddings)

# print(all_frames_path)
# print(all_frames_embeddings_2d)

uniform_frames_embeddings_2d = np.vstack([all_frames_embeddings_2d[idx] for idx in uniform_indices])
rag_frames_embeddings_2d_no_ft = np.vstack([all_frames_embeddings_2d[idx] for idx in rag_indices_no_ft])
rag_frames_embeddings_2d_sc = np.vstack([all_frames_embeddings_2d[idx] for idx in rag_indices_sc])
rag_frames_embeddings_2d_gc = np.vstack([all_frames_embeddings_2d[idx] for idx in rag_indices_gc])

plt.figure(figsize=(8, 5))
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.gca().spines['top'].set_color('black')
plt.gca().spines['right'].set_color('black')
plt.gca().tick_params(axis='x', colors='black')
plt.gca().tick_params(axis='y', colors='black')
plt.gca().title.set_color('black')
plt.gca().xaxis.label.set_color('black')
plt.gca().yaxis.label.set_color('black')

plt.scatter(text_embeddings_2d[:, 0], text_embeddings_2d[:, 1], s=150, marker='X', color='r', label='Question Embedding')
plt.scatter(all_frames_embeddings_2d[:, 0], all_frames_embeddings_2d[:, 1], s=10, color='gray', label='Frame Embeddings')
plt.scatter(uniform_frames_embeddings_2d[:, 0], uniform_frames_embeddings_2d[:, 1], s=100, facecolors='none', edgecolors='g', label='Uniform Sampling')
plt.scatter(rag_frames_embeddings_2d_no_ft[:, 0], rag_frames_embeddings_2d_no_ft[:, 1], s=100, facecolors='none', edgecolors='y', label='RAG-Adapter Sampling (No Fine-Tuning)')
# plt.scatter(rag_frames_embeddings_2d_sc[:, 0], rag_frames_embeddings_2d_sc[:, 1], s=100, facecolors='none', edgecolors='c')
plt.scatter(rag_frames_embeddings_2d_gc[:, 0], rag_frames_embeddings_2d_gc[:, 1], s=100, facecolors='none', edgecolors='b', label='RAG-Adapter Sampling (GCL Fine-Tuning)')

plt.gca().set_aspect('equal', 'datalim')
legend = plt.legend(loc='best', facecolor='none', edgecolor='black')
for text in legend.get_texts():
    text.set_color('black')
plt.title('2D Visualization of Question and Frame Embeddings')
plt.xlabel('Embeddings Dimension 1')
plt.ylabel('Embeddings Dimension 2')
plt.savefig("embedding.png", transparent=True)
plt.show()


# Determine the Necessary Information Frame (NIF) of the benchmark

## Determine the NIF of the Video-MME

In [ ]:
for video_id in picked:
    save_path = f"{DATA_ROOT}/NIE/full_q/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    # else:
    #     files = os.listdir(save_path)
    #     if len(files) == 3:
    #         continue

    if os.path.isdir(f"{WORK_ROOT}/qdrant_db/nie/indexs/{video_id}"):

        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/nif/clients/{video_id}")
        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )
        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, 
            image_store=image_store, 
            persist_dir=f"{WORK_ROOT}/qdrant_db/nie/indexs/{video_id}"
            )
        
        index = load_index_from_storage(storage_context=storage_context, embed_model=bge_m3,  image_embed_model=clip_model)
    else:
        frames = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}")
        frames.sort()

        captions = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/captions/{video_id}")
        captions.sort()
        
        # 1. Create a local Qdrant vector store
        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/nie/clients/{video_id}")

        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )

        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, image_store=image_store
        )

        # Create the MultiModal index
        files = []
        files.extend([f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{frame}" for frame in frames])
        files.extend([f"{DATA_ROOT}/dataset/Video-MME/captions/{video_id}/{caption}" for caption in captions])
        # print(files)
        documents = SimpleDirectoryReader(input_files=files).load_data()
    
        index = MultiModalVectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            embed_model=bge_m3_gc,
            image_embed_model=clip_gc
        )
        
        index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/nie/indexs/{video_id}")

    for question_id in questions[video_id]:

        frames_dir = os.path.join(save_path, question_id)
        if not os.path.exists(frames_dir):
            os.makedirs(frames_dir)
        
        print(f"processing: {video_id} {question_id}")
        question = questions[video_id][question_id]["question"]

        q_o = ""
        q_o += questions[video_id][question_id]["question"] + "\n"
        q_o += '\n'.join(questions[video_id][question_id]["options"]) + "\n"
        # q_o += "Answer: " + questions[video_id][question_id]["answer"]
        q_o_a = q_o + "Answer: " + questions[video_id][question_id]["answer"]
        # print(question)
        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(q_o[:77])

        caption_scores = defaultdict(int)
        image_scores = defaultdict(int)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
                # retrieved_image.append(res_node.node.metadata["file_path"])
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
        sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
        retrieved_image = []
        # print(image_scores)
        for k in sort[:50]:
            retrieved_image.append(k[0])
            
        # plot_images(retrieved_image)
        print(retrieved_image)
        for image in retrieved_image:
            new_image_path = frames_dir
            shutil.copy2(image, frames_dir)
        question_file = os.path.join(frames_dir, f"0_question.txt")
        with open(question_file, 'w') as f:
            f.write(q_o_a)


In [ ]:
# Calculate the NIF
with open(f"{DATA_ROOT}/NIF/Video-MME/video_mme_nif.csv", 'r', encoding='utf-8') as f:
    nie_mlvu = csv.reader(f)
    header = next(nie_mlvu)

    tc = 0
    lines = 0
    for row in nie_mlvu:
        fc = int(row[2])
        pos = row[3].split(";")
        tc += int(fc)
        lines += 1
        if fc != len(pos):
            print(row)
        if row[0] == "#NAME?":
            if row[1] in ['395-1','395-2','395-3']:
                print("-XpJeDGh8No")
            if row[1] in ['536-1','536-2','536-3']:
                print("-c8eATXUui8")
    
print("Video-MME NIF: ", tc/lines) 



## Determine Recall@10 of Video-MME

In [ ]:
frames_type = ["10_frames_no_ft", "10_frames_sc", "10_frames_gc", "10_frames_gc_only_caption", "10_frames_gc_only_image", "10_frames_gc_without_dual_ranker"]
with open(f"{DATA_ROOT}/NIF/Video-MME/video_mme_nif.csv", 'r', encoding='utf-8') as f:
    nie_mlvu = csv.reader(f)
    header = next(nie_mlvu)

    index = 2
    for row in nie_mlvu:
        video_id = row[0]
        q_id = row[1]
        # print(video_id)
        if video_id == "#NAME?":
            if q_id in ['395-1','395-2','395-3']:
                video_id = "-XpJeDGh8No"
            if q_id in ['536-1','536-2','536-3']:
                video_id = "-c8eATXUui8"
        
        frames = os.listdir(f"{DATA_ROOT}/rag_adapter_sampled_frames/Video-MME/recall/{frames_type[5]}/{video_id}/{q_id}")
        frames.sort()
        frames_index = [int(frame.split(".")[0]) for frame in frames]
        # print(video_id, q_id, frames_index)
        fc = int(row[2])
        pos = row[3].split(";")
        pos_index = [int(p) for p in pos]
        # print(pos_index)
        recall = 0
        for p in pos_index:
            t = p + 2
            if t in frames_index:
                recall += 1
        print(recall)
        # print(index, recall)
        # if index % 3 == 1:
        #     print(" ")
        # index += 1


## Determine the NIF of the Perception Test

In [ ]:
# Calculate the NIF
with open(f"{DATA_ROOT}/NIF/Perception Test/perception_test_nif.csv", 'r', encoding='utf-8') as f:
    nie_perception = csv.reader(f)
    header = next(nie_perception)

    tc = 0
    lines = 0
    for row in nie_perception:
        fc = int(row[2])
        pos = row[3].split(";")
        tc += int(fc)
        lines += 1
        if fc != len(pos):
            print(row)
    
print("Perception Test NIF: ", tc/lines) 


## Determine the NIF of the Egoschema

In [ ]:
# Calculate the NIF
with open(f"{DATA_ROOT}/NIF/Egochema/egoschema_nif.csv", 'r', encoding='utf-8') as f:
    nie_perception = csv.reader(f)
    header = next(nie_perception)

    tc = 0
    lines = 0
    for row in nie_perception:
        fc = int(row[1])
        pos = row[2].split(";")
        tc += int(fc)
        lines += 1
        if fc != len(pos):
            print(row)
    
print("Egoschema NIF: ", tc/lines) 


## Determine the NIF of MLVU

In [ ]:
scores = defaultdict(list)
for video_id in questions.keys():
    task = questions[video_id]["question_type"]
    save_path = f"{DATA_ROOT}/NIF/MLVU/only_q/{task}/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    if os.path.isdir(f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/indexs/{video_id}"):
        
        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/clients/{video_id}")
        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )
        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, 
            image_store=image_store, 
            persist_dir=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/indexs/{video_id}"
            )
        
        index = load_index_from_storage(storage_context=storage_context, embed_model=bge_m3_gc,  image_embed_model=clip_gc)
    else:
        frames = os.listdir(f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}")
        frames.sort()

        captions = os.listdir(f"{DATA_ROOT}/dataset/MLVU/captions/{task}/{video_id}")
        captions.sort()
        
        # 1. Create a local Qdrant vector store
        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/clients/{video_id}")

        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )

        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, image_store=image_store
        )

        # Create the MultiModal index
        files = []
        files.extend([f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}/{frame}" for frame in frames])
        files.extend([f"{DATA_ROOT}/dataset/MLVU/captions/{task}/{video_id}/{caption}" for caption in captions])
        # print(files)
        
        documents = SimpleDirectoryReader(input_files=files).load_data()
    
        index = MultiModalVectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            embed_model=bge_m3_gc,
            image_embed_model=clip_gc
        )
        
        index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/indexs/{video_id}")

    for i, question in enumerate(questions[video_id]["question"]):
        print(f"processing: {video_id} {question}")
        frames_dir = os.path.join(save_path, question[:40])
        if not os.path.exists(frames_dir):
            os.makedirs(frames_dir)
        else:
            files = os.listdir(frames_dir)
            if len(files) == 51:
                continue
        
        print(f"processing: {video_id} {question}")

        q_o = ""
        q_o += question + "\n"
        if "candidates" in questions[video_id] and len(questions[video_id]["candidates"]) != 0:
            candidates = questions[video_id]["candidates"][i]
            for c in candidates:
                q_o += c + "\n"
        q_o += "Answer: " + questions[video_id]["answer"][i]
        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(question[:77])

        caption_scores = defaultdict(int)
        image_scores = defaultdict(int)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
        retrieved_image = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=50)
        print(retrieved_image)
        for image in retrieved_image:
            new_image_path = frames_dir
            shutil.copy2(image, frames_dir)
        question_file = os.path.join(frames_dir, f"0_question.txt")
        with open(question_file, 'w') as f:
            f.write(q_o)


In [ ]:
## Calculate the NIF
with open(f"{DATA_ROOT}/NIF/MLVU/mlvu_nif.csv", 'r', encoding='utf-8') as f:
    nie_mlvu = csv.reader(f)
    header = next(nie_mlvu)

    tc = 0
    lines = 0
    for row in nie_mlvu:
        fc = int(row[3])
        pos = row[4].split(";")
        tc += int(fc)
        lines += 1
        if fc != len(pos):
            print(row)
    
print("MLVU NIF: ", tc/lines) 


# Compute ASS metrics for the benchmark

## Determine Video-MME's ASS


In [ ]:
scores = defaultdict(list)
for video_id in picked:
    # save_path = f"{DATA_ROOT}/rag_adapter_sampled_frames/Video-MME/10_frames_no_ft/{video_id}/"
    # if not os.path.exists(save_path):
    #     os.makedirs(save_path)
    # else:
    #     files = os.listdir(save_path)
    #     if len(files) == 3:
    #         continue

    frames = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}")
    frames.sort()

    captions = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/captions/{video_id}")
    captions.sort()
    
    # 1. Create a local Qdrant vector store
    # q_client = qdrant_client.QdrantClient(location=":memory:")
    q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/gc_rag/clients/{video_id}")

    text_store = QdrantVectorStore(
        client=q_client, collection_name="text_collection"
    )

    image_store = QdrantVectorStore(
        client=q_client, collection_name="image_collection"
    )

    storage_context = StorageContext.from_defaults(
        vector_store=text_store, image_store=image_store
    )

    # Create the MultiModal index
    files = []
    files.extend([f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{frame}" for frame in frames])
    files.extend([f"{DATA_ROOT}/dataset/Video-MME/captions/{video_id}/{caption}" for caption in captions])
    # print(files)
    documents = SimpleDirectoryReader(input_files=files).load_data()

    index = MultiModalVectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        embed_model=bge_m3_gc,
        image_embed_model=clip_model
    )
    
    index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/gc_rag/indexs/{video_id}")

    for question_id in questions[video_id]:
        # frames_dir = os.path.join(save_path, question_id)
        # if not os.path.exists(frames_dir):
        #     os.makedirs(frames_dir)
        
        print(f"processing: {video_id} {question_id}")
        question = questions[video_id][question_id]["question"]

        # q_o = ""
        # q_o += questions[video_id][question_id]["question"] + "\n"
        # q_o += '\n'.join(questions[video_id][question_id]["options"]) + "\n"
        # q_o += "Answer: " + questions[video_id][question_id]["answer"]
        # q_o_a = q_o + "Answer: " + questions[video_id][question_id]["answer"]
        # print(question)
        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(question)

        caption_scores = defaultdict(float)
        image_scores = defaultdict(float)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
                # retrieved_image.append(res_node.node.metadata["file_path"])
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
        # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
        # retrieved_image = []
        # for k in sort[:10]:
        #     retrieved_image.append(k[0])
        print("image_scores", len(image_scores))
        if len(image_scores) > 10:
            selected_items_10 = mmr_selection(image_scores, lambda_param=0.7, top_k=10)
        else:
            selected_items_10 = list(image_scores.keys())
        if len(image_scores) > 30:
            selected_items_30 = mmr_selection(image_scores, lambda_param=0.7, top_k=30)
        else:
            selected_items_30 = list(image_scores.keys())
        if len(image_scores) > 50:
            selected_items_50 = mmr_selection(image_scores, lambda_param=0.7, top_k=50)
        else:
            selected_items_50 = list(image_scores.keys())

        for image in selected_items_10:
            scores["10"].append(image_scores[image])
        for image in selected_items_30:
            scores["30"].append(image_scores[image])
        for image in selected_items_50:
            scores["50"].append(image_scores[image])
print(scores)
np.save(os.path.join(f"{WORK_ROOT}", "scores.npy"), scores)

scores = np.load(f"{WORK_ROOT}/scores.npy", allow_pickle=True).item()
print(scores)
for k, v in scores.items():
    print(f"Top {k}: {np.mean(v)}")


## Determine MLVU's ASS

In [ ]:
scores = defaultdict(list)
for video_id in questions.keys():
    task = questions[video_id]["question_type"]
    frames = os.listdir(f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}")
    frames.sort()

    captions = os.listdir(f"{DATA_ROOT}/dataset/MLVU/captions/{task}/{video_id}")
    captions.sort()
    
    # 1. Create a local Qdrant vector store
    # q_client = qdrant_client.QdrantClient(location=":memory:")
    q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/clients/{video_id}")

    text_store = QdrantVectorStore(
        client=q_client, collection_name="text_collection"
    )

    image_store = QdrantVectorStore(
        client=q_client, collection_name="image_collection"
    )

    storage_context = StorageContext.from_defaults(
        vector_store=text_store, image_store=image_store
    )

    # Create the MultiModal index
    files = []
    files.extend([f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}/{frame}" for frame in frames])
    files.extend([f"{DATA_ROOT}/dataset/MLVU/captions/{task}/{video_id}/{caption}" for caption in captions])
    # print(files)
    documents = SimpleDirectoryReader(input_files=files).load_data()

    index = MultiModalVectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        embed_model=bge_m3_gc,
        image_embed_model=clip_gc
    )
    
    index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/indexs/{video_id}")

    for question in questions[video_id]["question"]:
        # frames_dir = os.path.join(save_path, question_id)
        # if not os.path.exists(frames_dir):
        #     os.makedirs(frames_dir)
        
        print(f"processing: {video_id} {question}")

        # q_o = ""
        # q_o += questions[video_id][question_id]["question"] + "\n"
        # q_o += '\n'.join(questions[video_id][question_id]["options"]) + "\n"
        # q_o += "Answer: " + questions[video_id][question_id]["answer"]
        # q_o_a = q_o + "Answer: " + questions[video_id][question_id]["answer"]
        # print(question)
        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(question)

        caption_scores = defaultdict(float)
        image_scores = defaultdict(float)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
                # retrieved_image.append(res_node.node.metadata["file_path"])
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
        # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
        # retrieved_image = []
        # for k in sort[:10]:
        #     retrieved_image.append(k[0])
        print("image_scores", len(image_scores))
        if len(image_scores) > 10:
            selected_items_10 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=10)
        else:
            selected_items_10 = list(image_scores.keys())
        if len(image_scores) > 30:
            selected_items_30 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=30)
        else:
            selected_items_30 = list(image_scores.keys())
        if len(image_scores) > 50:
            selected_items_50 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=50)
        else:
            selected_items_50 = list(image_scores.keys())

        for image in selected_items_10:
            scores["10"].append(image_scores[image])
        for image in selected_items_30:
            scores["30"].append(image_scores[image])
        for image in selected_items_50:
            scores["50"].append(image_scores[image])
print(scores)
np.save(os.path.join(f"{WORK_ROOT}", "mlvu_scores.npy"), scores)


In [ ]:
scores = np.load(f"{WORK_ROOT}/mlvu_scores.npy", allow_pickle=True).item()
print(scores)
for k, v in scores.items():
    print(f"Top {k}: {np.mean(v)}")


## Determine Percetion_Test's ASS

In [ ]:
scores = defaultdict(list)
for video_id in questions_perception_test.keys():

    frames = os.listdir(f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}")
    frames.sort()

    captions = os.listdir(f"{DATA_ROOT}/dataset/Perception_Test/captions/{video_id}")
    captions.sort()
    
    # 1. Create a local Qdrant vector store
    # q_client = qdrant_client.QdrantClient(location=":memory:")
    q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/gc_rag/clients/{video_id}")

    text_store = QdrantVectorStore(
        client=q_client, collection_name="text_collection"
    )

    image_store = QdrantVectorStore(
        client=q_client, collection_name="image_collection"
    )

    storage_context = StorageContext.from_defaults(
        vector_store=text_store, image_store=image_store
    )

    # Create the MultiModal index
    files = []
    files.extend([f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}/{frame}" for frame in frames])
    files.extend([f"{DATA_ROOT}/dataset/Perception_Test/captions/{video_id}/{caption}" for caption in captions])
    # print(files)
    documents = SimpleDirectoryReader(input_files=files).load_data()

    index = MultiModalVectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        embed_model=bge_m3_gc,
        image_embed_model=clip_gc
    )
    
    index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/gc_rag/indexs/{video_id}")

    for i, q in enumerate(questions_perception_test[video_id]["question"]):
        
        print(f"processing: {video_id} {i}_th question")
        question = q

        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(question)

        caption_scores = defaultdict(float)
        image_scores = defaultdict(float)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
                # retrieved_image.append(res_node.node.metadata["file_path"])
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
        # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
        # retrieved_image = []
        # for k in sort[:10]:
        #     retrieved_image.append(k[0])
        print("image_scores", len(image_scores))
        if len(image_scores) > 10:
            selected_items_10 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=10)
        else:
            selected_items_10 = list(image_scores.keys())
        if len(image_scores) > 30:
            selected_items_30 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=30)
        else:
            selected_items_30 = list(image_scores.keys())
        if len(image_scores) > 50:
            selected_items_50 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=50)
        else:
            selected_items_50 = list(image_scores.keys())

        for image in selected_items_10:
            scores["10"].append(image_scores[image])
        for image in selected_items_30:
            scores["30"].append(image_scores[image])
        for image in selected_items_50:
            scores["50"].append(image_scores[image])
# print(scores)
np.save(os.path.join(f"{WORK_ROOT}", "perception_test_scores.npy"), scores)


In [ ]:
scores = np.load(f"{WORK_ROOT}/perception_test_scores.npy", allow_pickle=True).item()
# print(scores)
for k, v in scores.items():
    print(f"Top {k}: {np.mean(v)}")


## Determine Egochema's ASS

In [ ]:
# runtime: 21m51.8s
scores = defaultdict(list)
for video_id in questions_egoschema.keys():

    frames = os.listdir(f"{DATA_ROOT}/dataset/EgoSchema/frames/{video_id}")
    frames.sort()

    captions = os.listdir(f"{DATA_ROOT}/dataset/EgoSchema/captions/{video_id}")
    captions.sort()
    
    # 1. Create a local Qdrant vector store
    # q_client = qdrant_client.QdrantClient(location=":memory:")
    q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/gc_rag/clients/{video_id}")

    text_store = QdrantVectorStore(
        client=q_client, collection_name="text_collection"
    )

    image_store = QdrantVectorStore(
        client=q_client, collection_name="image_collection"
    )

    storage_context = StorageContext.from_defaults(
        vector_store=text_store, image_store=image_store
    )

    # Create the MultiModal index
    files = []
    files.extend([f"{DATA_ROOT}/dataset/EgoSchema/frames/{video_id}/{frame}" for frame in frames])
    files.extend([f"{DATA_ROOT}/dataset/EgoSchema/captions/{video_id}/{caption}" for caption in captions])
    # print(files)
    documents = SimpleDirectoryReader(input_files=files).load_data()

    index = MultiModalVectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        embed_model=bge_m3_gc,
        image_embed_model=clip_gc
    )
    
    index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/gc_rag/indexs/{video_id}")

    for i, q in enumerate(questions_egoschema[video_id]["question"]):
        
        print(f"processing: {video_id} {i}_th question")
        question = q

        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(question)

        caption_scores = defaultdict(float)
        image_scores = defaultdict(float)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
                # retrieved_image.append(res_node.node.metadata["file_path"])
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/EgoSchema/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
        # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
        # retrieved_image = []
        # for k in sort[:10]:
        #     retrieved_image.append(k[0])
        print("image_scores", len(image_scores))
        if len(image_scores) > 10:
            selected_items_10 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=10)
        else:
            selected_items_10 = list(image_scores.keys())
        if len(image_scores) > 30:
            selected_items_30 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=30)
        else:
            selected_items_30 = list(image_scores.keys())
        if len(image_scores) > 50:
            selected_items_50 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=50)
        else:
            selected_items_50 = list(image_scores.keys())

        for image in selected_items_10:
            scores["10"].append(image_scores[image])
        for image in selected_items_30:
            scores["30"].append(image_scores[image])
        for image in selected_items_50:
            scores["50"].append(image_scores[image])

np.save(os.path.join(f"{WORK_ROOT}", "egoschema_scores.npy"), scores)


In [ ]:
scores = np.load(f"{WORK_ROOT}/egoschema_scores.npy", allow_pickle=True).item()
# print(scores)
for k, v in scores.items():
    print(f"Top {k}: {np.mean(v)}")


# Use the RAG-Adapter to pre-store relevant frames (Top5, Top10, Top20, etc.)

## Store Video-MME related frames

In [ ]:
import time
for video_id in questions_video_mme.keys():
    save_path = f"{DATA_ROOT}/rag_adapter_sampled_frames/Video-MME/10_frames_gc_with_time/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    # 记录开始时间
    start_time = time.time()

    if os.path.isdir(f"{WORK_ROOT}/qdrant_db/video-mme/indexs/{video_id}"):
        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/video-mme/clients/{video_id}")
        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )
        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, 
            image_store=image_store, 
            persist_dir=f"{WORK_ROOT}/qdrant_db/video-mme/indexs/{video_id}"
            )
        
        index = load_index_from_storage(storage_context=storage_context, embed_model=bge_m3_gc,  image_embed_model=clip_gc)
    else:
        frames = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}")
        frames.sort()

        captions = os.listdir(f"{DATA_ROOT}/dataset/Video-MME/captions/{video_id}")
        captions.sort()
        
        # 1. Create a local Qdrant vector store
        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/video-mme/clients/{video_id}")

        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )

        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, image_store=image_store
        )

        # Create the MultiModal index
        files = []
        files.extend([f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{frame}" for frame in frames])
        files.extend([f"{DATA_ROOT}/dataset/Video-MME/captions/{video_id}/{caption}" for caption in captions])
        
        # print(files)
        documents = SimpleDirectoryReader(input_files=files).load_data()
    
        index = MultiModalVectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            embed_model=bge_m3_gc,
            image_embed_model=clip_gc
        )
        
        index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/video-mme/indexs/{video_id}")

    for question_id in questions_video_mme[video_id]:
        frames_dir = os.path.join(save_path, question_id)
        if not os.path.exists(frames_dir):
            os.makedirs(frames_dir)
        
        print(f"processing: {video_id} {question_id}")
        question = questions_video_mme[video_id][question_id]["question"]

        # q_o = ""
        # q_o += questions[video_id][question_id]["question"] + "\n"
        # q_o += '\n'.join(questions[video_id][question_id]["options"]) + "\n"
        # q_o += "Answer: " + questions[video_id][question_id]["answer"]
        # q_o_a = q_o + "Answer: " + questions[video_id][question_id]["answer"]
        # print(question)

        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(question)

        caption_scores = defaultdict(float)
        image_scores = defaultdict(float)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/Video-MME/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()

        # Do not use dual ranker
        # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
        # print(f"sort: {sort}")
        # for k, v in sort[:10]:
        #     shutil.copy2(k, frames_dir)

        # Use Dual Ranker
        if len(image_scores) > 10:
            selected_items_10 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=10)
        else:
            selected_items_10 = list(image_scores.keys())

        for image in selected_items_10:
            shutil.copy2(image, frames_dir)

    # 计算执行时间
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"执行时间: {execution_time:.4f} 秒")

    with open(f"{save_path}/running_time.txt", "w") as f_time:
            f_time.write(f"Execution time: {execution_time:.4f} seconds\n")


## Store Perception Test related frames

In [ ]:
# runtime: 8m43.2s
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Perception_Test/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]

            save_path = f"{DATA_ROOT}/rag_adapter_sampled_frames/Perception_Test/10_frames_gc/{video_id}/"
            if not os.path.exists(save_path):
                os.makedirs(save_path)
            # else:
            #     files = os.listdir(save_path)
            #     if len(files) == 3:
            #         continue

            if os.path.isdir(f"{WORK_ROOT}/qdrant_db/perception_test/10_frames_gc/indexs/{video_id}"):

                # q_client = qdrant_client.QdrantClient(location=":memory:")
                q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/perception_test/10_frames_gc/clients/{video_id}")
                text_store = QdrantVectorStore(
                    client=q_client, collection_name="text_collection"
                )
                image_store = QdrantVectorStore(
                    client=q_client, collection_name="image_collection"
                )

                storage_context = StorageContext.from_defaults(
                    vector_store=text_store, 
                    image_store=image_store, 
                    persist_dir=f"{WORK_ROOT}/qdrant_db/perception_test/10_frames_gc/indexs/{video_id}"
                    )
                
                index = load_index_from_storage(storage_context=storage_context, embed_model=bge_m3_gc,  image_embed_model=clip_gc)
            else:
                frames = os.listdir(f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}")
                frames.sort()

                captions = os.listdir(f"{DATA_ROOT}/dataset/Perception_Test/captions/{video_id}")
                captions.sort()
                
                # 1. Create a local Qdrant vector store
                # q_client = qdrant_client.QdrantClient(location=":memory:")
                q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/perception_test/10_frames_gc/clients/{video_id}")

                text_store = QdrantVectorStore(
                    client=q_client, collection_name="text_collection"
                )

                image_store = QdrantVectorStore(
                    client=q_client, collection_name="image_collection"
                )

                storage_context = StorageContext.from_defaults(
                    vector_store=text_store, image_store=image_store
                )

                # Create the MultiModal index
                files = []
                files.extend([f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}/{frame}" for frame in frames])
                files.extend([f"{DATA_ROOT}/dataset/Perception_Test/captions/{video_id}/{caption}" for caption in captions])
                
                # print(files)
                documents = SimpleDirectoryReader(input_files=files).load_data()
            
                index = MultiModalVectorStoreIndex.from_documents(
                    documents,
                    storage_context=storage_context,
                    embed_model=bge_m3_gc,
                    image_embed_model=clip_gc
                )
                
                index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/perception_test/10_frames_gc/indexs/{video_id}")

            for i in range(len(questions_perception_test[video_id]["question"])):
                question_id = questions_perception_test[video_id]["id"][i]
                print(f"processing: {video_id} {question_id}")

                frames_dir = os.path.join(save_path, str(question_id))
                if not os.path.exists(frames_dir):
                    os.makedirs(frames_dir)
                
                question = questions_perception_test[video_id]["question"][i]

                retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
                retrieval_results = retriever.retrieve(question)

                caption_scores = defaultdict(float)
                image_scores = defaultdict(float)
                for res_node in retrieval_results:
                    if isinstance(res_node.node, ImageNode):
                        image_index = res_node.node.metadata['file_name'].split(".")[0]
                        image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
                        # retrieved_image.append(res_node.node.metadata["file_path"])
                    else:
                        image_index = res_node.node.metadata['file_name'].split(".")[0]
                        image_scores[f"{DATA_ROOT}/dataset/Perception_Test/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()
                
                # print(f"image_scores: {image_scores}")
                # Do not use dual ranker
                # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
                # print(f"sort: {sort}")
                # for k, v in sort[:10]:
                #     shutil.copy2(k, frames_dir)

                # Use Dual Ranker
                if len(image_scores) > 10:
                    selected_items_10 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=10)
                else:
                    selected_items_10 = list(image_scores.keys())

                for image in selected_items_10:
                    shutil.copy2(image, frames_dir)


## Store Egochema related frames

In [ ]:
# runtime: 8m54.6s
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/EgoSchema/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]

            save_path = f"{DATA_ROOT}/rag_adapter_sampled_frames/Egoschema/10_frames_gc/{video_id}/"
            if not os.path.exists(save_path):
                os.makedirs(save_path)
            # else:
            #     files = os.listdir(save_path)
            #     if len(files) == 3:
            #         continue

            if os.path.isdir(f"{WORK_ROOT}/qdrant_db/egohema/10_frames_gc/indexs/{video_id}"):

                # q_client = qdrant_client.QdrantClient(location=":memory:")
                q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/egohema/10_frames_gc/clients/{video_id}")
                text_store = QdrantVectorStore(
                    client=q_client, collection_name="text_collection"
                )
                image_store = QdrantVectorStore(
                    client=q_client, collection_name="image_collection"
                )

                storage_context = StorageContext.from_defaults(
                    vector_store=text_store, 
                    image_store=image_store, 
                    persist_dir=f"{WORK_ROOT}/qdrant_db/egohema/10_frames_gc/indexs/{video_id}"
                    )
                
                index = load_index_from_storage(storage_context=storage_context, embed_model=bge_m3_gc,  image_embed_model=clip_gc)
            else:
                frames = os.listdir(f"{DATA_ROOT}/dataset/EgoSchema/frames/{video_id}")
                frames.sort()

                captions = os.listdir(f"{DATA_ROOT}/dataset/EgoSchema/captions/{video_id}")
                captions.sort()
                
                # 1. Create a local Qdrant vector store
                # q_client = qdrant_client.QdrantClient(location=":memory:")
                q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/egohema/10_frames_gc/clients/{video_id}")

                text_store = QdrantVectorStore(
                    client=q_client, collection_name="text_collection"
                )

                image_store = QdrantVectorStore(
                    client=q_client, collection_name="image_collection"
                )

                storage_context = StorageContext.from_defaults(
                    vector_store=text_store, image_store=image_store
                )

                # Create the MultiModal index
                files = []
                files.extend([f"{DATA_ROOT}/dataset/EgoSchema/frames/{video_id}/{frame}" for frame in frames])
                files.extend([f"{DATA_ROOT}/dataset/EgoSchema/captions/{video_id}/{caption}" for caption in captions])
                
                # print(files)
                documents = SimpleDirectoryReader(input_files=files).load_data()
            
                index = MultiModalVectorStoreIndex.from_documents(
                    documents,
                    storage_context=storage_context,
                    embed_model=bge_m3_gc,
                    image_embed_model=clip_gc
                )
                
                index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/egohema/10_frames_gc/indexs/{video_id}")

            for i in range(len(questions_egoschema[video_id]["question"])):
                print(f"processing: {video_id}")

                frames_dir = os.path.join(save_path)
                if not os.path.exists(frames_dir):
                    os.makedirs(frames_dir)
                
                question = questions_egoschema[video_id]["question"][i]

                retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
                retrieval_results = retriever.retrieve(question)

                caption_scores = defaultdict(float)
                image_scores = defaultdict(float)
                for res_node in retrieval_results:
                    if isinstance(res_node.node, ImageNode):
                        image_index = res_node.node.metadata['file_name'].split(".")[0]
                        image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
                        # retrieved_image.append(res_node.node.metadata["file_path"])
                    else:
                        image_index = res_node.node.metadata['file_name'].split(".")[0]
                        image_scores[f"{DATA_ROOT}/dataset/EgoSchema/frames/{video_id}/{image_index}.jpg"] += res_node.get_score()
                
                # print(f"image_scores: {image_scores}")
                # Do not use dual ranker
                # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
                # print(f"sort: {sort}")
                # for k, v in sort[:10]:
                #     shutil.copy2(k, frames_dir)

                # Use Dual Ranker
                if len(image_scores) > 10:
                    selected_items_10 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=10)
                else:
                    selected_items_10 = list(image_scores.keys())

                for image in selected_items_10:
                    shutil.copy2(image, frames_dir)


## Store MLVU related frames

In [ ]:
for video_id in questions.keys():
    task = questions[video_id]["question_type"]
    save_path = f"{DATA_ROOT}/rag_adapter_sampled_frames/MLVU/20_frames_gc/{task}/{video_id}/"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    # else:
    #     files = os.listdir(save_path)
    #     if len(files) == 3:
    #         continue

    if os.path.isdir(f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/indexs/{video_id}"):

        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/clients/{video_id}")
        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )
        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, 
            image_store=image_store, 
            persist_dir=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/indexs/{video_id}"
            )
        
        index = load_index_from_storage(storage_context=storage_context, embed_model=bge_m3_gc,  image_embed_model=clip_gc)
    else:
        frames = os.listdir(f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}")
        frames.sort()

        captions = os.listdir(f"{DATA_ROOT}/dataset/MLVU/captions/{task}/{video_id}")
        captions.sort()
        
        # 1. Create a local Qdrant vector store
        # q_client = qdrant_client.QdrantClient(location=":memory:")
        q_client = qdrant_client.QdrantClient(path=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/clients/{video_id}")

        text_store = QdrantVectorStore(
            client=q_client, collection_name="text_collection"
        )

        image_store = QdrantVectorStore(
            client=q_client, collection_name="image_collection"
        )

        storage_context = StorageContext.from_defaults(
            vector_store=text_store, image_store=image_store
        )

        # Create the MultiModal index
        files = []
        files.extend([f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}/{frame}" for frame in frames])
        files.extend([f"{DATA_ROOT}/dataset/MLVU/captions/{task}/{video_id}/{caption}" for caption in captions])
        # print(files)
        documents = SimpleDirectoryReader(input_files=files).load_data()
    
        index = MultiModalVectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            embed_model=bge_m3_gc,
            image_embed_model=clip_gc
        )
        
        index.storage_context.persist(persist_dir=f"{WORK_ROOT}/qdrant_db/mlvu/gc_rag/indexs/{video_id}")

    for question in questions[video_id]["question"]:
        frames_dir = os.path.join(save_path, question[:40])
        if not os.path.exists(frames_dir):
            os.makedirs(frames_dir)
        
        print(f"processing: {video_id} {question}")

        # q_o = ""
        # q_o += question + "\n"
        # if "candidates" in questions[video_id] and len(questions[video_id]["candidates"]) != 0:
        #     candidates = questions[video_id]["candidates"][i]
        #     for c in candidates:
        #         q_o += c + "\n"
        # q_o += "Answer: " + questions[video_id]["answer"][i]

        retriever = index.as_retriever(similarity_top_k=50, image_similarity_top_k=50)
        retrieval_results = retriever.retrieve(question[:77])

        caption_scores = defaultdict(float)
        image_scores = defaultdict(float)
        for res_node in retrieval_results:
            if isinstance(res_node.node, ImageNode):
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[res_node.node.metadata["file_path"]] += res_node.get_score()
            else:
                image_index = res_node.node.metadata['file_name'].split(".")[0]
                image_scores[f"{DATA_ROOT}/dataset/MLVU/frames/{task}/{video_id}/{image_index}.jpg"] += res_node.get_score()
        
        # sort = sorted(image_scores.items(), key=lambda x: x[1], reverse=True)
        print(len(image_scores))
        print(image_scores)
        if len(image_scores) > 20:
            selected_items_20 = mmr_selection(image_scores, bge_m3_gc, clip_gc, lambda_param=0.7, top_k=20)
        else:
            selected_items_20 = list(image_scores.keys())

        for image in selected_items_20:
            shutil.copy2(image, frames_dir)


# Use the RAG-Adapter to get the most relevant context from the srt file

In [ ]:
# Sort srts by time
def extract_start_time(time_string):
    start_time_str = time_string.split(" to ")[0].replace("From ", "")
    return datetime.datetime.strptime(start_time_str, "%H:%M:%S.%f").time()

for video_id in picked:
    if not os.path.exists(f"{DATA_ROOT}/dataset/Video-MME/subtitle/{video_id}.srt"):
        print(f"{video_id}'s srt does not exist.")
        continue
    save_path = f"{DATA_ROOT}/rag_adapter_sampled_frames/Video-MME/10_srt_gc/{video_id}"
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    print(f"Process {video_id}")
    subs = pysrt.open(f"{DATA_ROOT}/dataset/Video-MME/subtitle/{video_id}.srt")
    documents = []
    for sub in subs:
        start_time = sub.start.to_time().strftime('%H:%M:%S.%f')[:-3]
        end_time = sub.end.to_time().strftime('%H:%M:%S.%f')[:-3]

        content = re.sub(r'<[^>]+>', '', sub.text)
        text = f"From {start_time} to {end_time}: {content}"
        documents.append(Document(text=text))

    index = VectorStoreIndex.from_documents(documents, embed_model=bge_m3_gc)
    retriever = index.as_retriever(similarity_top_k=10)
    for question_id in questions[video_id]:
        frames_dir = os.path.join(save_path, question_id)
        if not os.path.exists(frames_dir):
            os.makedirs(frames_dir)

        question = ""
        question += questions[video_id][question_id]["question"] + "\n"
        question += '\n'.join(questions[video_id][question_id]["options"]) + "\n"
       
        retrieval_results = retriever.retrieve(question)
        srt_contents = []
        for res_node in retrieval_results:
            srt_contents.append(res_node.node.get_text())
        srt_contents = sorted(srt_contents, key=extract_start_time)
  
        with open(os.path.join(frames_dir, "relevant_srt.txt"), "w+", encoding="utf-8") as f:
            for time_string in srt_contents:
                f.write(time_string + "\n")
